# 16 · Gradient-Matched Temporal Label Reliability — **pre-GPU only**

Hiện thực `FINAL_PLAN_V3_3_LABEL_RELIABILITY.md`. Ghi chép quyết định:
`REVIEW_PLAN_V3_3_FINAL.md` (5/5 thắc mắc đã chốt) và `REVIEW_PLAN_V3_2_GRADIENT_MATCHED.md`
(§3-bis: lỗi CLS ở nb14/15).

**Notebook này KHÔNG train.** Nó chạy đúng bốn bước tiền-GPU của plan §13:

```
0a  raw labels + coverage + temporal manifest + concentration   (C2, C3, C4)
0b  exact initial ASL logit-gradient share + beta_c             (C8)
0c  200-step smoke: implementation / runtime / stochasticity     (C9)
0d  deterministic calibration diagnostics                        (C8 + C9)
```

Lý do dừng ở đây: **§4.4 của plan có thể kết thúc thí nghiệm ngay tại C4.** Nếu
data-sufficiency gate fail thì `R2` mất primary status và **không** được sửa manifest rồi
tiếp tục. Viết train loop trước khi biết là lãng phí.

## Ba arm (khoá)

| arm | ordinary blank | contested blank | mọi ô khác |
|---|---:|---:|---|
| `R0` historical | 1 | 1 | 1 |
| `R1GM` uniform control | `β_c` | `β_c` | 1 |
| `R2` temporal targeting | 1 | **η = 0.10** | 1 |

`β_c` **suy ra từ** `η` (§1 plan) nên control luôn tự khớp initial logit-gradient mass của
chính mức can thiệp đang dùng:

$$\beta_c=\frac{\eta\,G_{\text{cont},c}+G_{\text{ord},c}}{G_{\text{cont},c}+G_{\text{ord},c}}$$

`NonTarget-10` tự động có `β = 1` vì `G_cont = 0`.

## Bốn chỗ khác nb14/15 — đọc kỹ

**1. Patch-only.** `patches = backbone.layernorm(h)[:, 1:]` — **bỏ CLS**. nb14/15 đưa cả CLS
vào ML-Decoder, lệch với H5 (train trên 1,369 patch token) và với nb11/13. Xem
`REVIEW_PLAN_V3_2` §3-bis.

**2. Identity gate ba tầng** so với `ArmModel(plain=True)` **nguyên bản nb13**, code path độc
lập, không dùng chung module. Cổng `G2` của nb14 so hai instance của **cùng** implementation
nên không bắt được lỗi CLS.

**3. Partial fine-tuning thật.** Blocks 9–11 **trainable**, bản sao riêng mỗi arm. Không notebook
nào trước đây train block backbone.

**4. Chuẩn hoá loss bằng hằng số** `B_actual × C`, **không** chia `Σw` — nếu chia `Σw` thì 10 nhãn
non-target nhận gradient lệch ~20% giữa các arm chỉ vì mẫu số khác.

## Không dùng

```
report text · expert relabel · hard relabel · pair-consistency loss · EHR · WASL/DBL
```

## C0 · Config khoá (prereg)

In [ ]:
import os, re, json, math, time, copy, pickle, random, shutil, tarfile, contextlib
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
LOCAL_IMG = os.environ.get('LOCAL_IMG', '/content/cxr_local')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')            # 04b LP-FT
H5_CKPT   = os.path.join(DATA_DIR, 'token_fusion', 'checkpoints', 'H5_s42_best.pt')   # 06b head
SC_DIR    = os.path.join(DATA_DIR, 'label_reliability')
for _d in (SC_DIR, f'{SC_DIR}/metrics', f'{SC_DIR}/checkpoints', f'{SC_DIR}/snapshots'):
    os.makedirs(_d, exist_ok=True)
PROJECT, CXR = 'mimic-ehr-v310', 'physionet-data.mimic_cxr_jpg'      # nguon nhan tho
RAW_PQ = os.path.join(OUT_DIR, 'cohort_raw_labels.parquet')          # FILE MOI, khong ghi de

# ---------------- KHOA: mot seed moi run ----------------
SEED = 42                       # doi thanh 1337 / 2024 cho seed xac nhan (§11 plan)

# ---------------- KHOA: arm & weight policy ----------------
ARMS  = ['R0', 'R1GM', 'R2']
ETA   = 0.10                    # weight cua contested blank trong R2
RARE3 = ['Pleural_Other', 'Fracture', 'Lung_Lesion']                 # khoa tu class_freq.pkl
WORST3_FIXED = ['Fracture', 'Pleural_Other', 'Enlarged_Cardiomediastinum']

# ---------------- KHOA: temporal manifest ----------------
WINDOW_DAYS   = 7               # +/- 7 ngay, persistence prior cho structural label
ANCHOR_RAW    = 1               # chi raw +1 lam anchor; KHONG dung raw -1
REQUIRE_SAME_VIEW = True
REQUIRE_SAME_ENCOUNTER = False  # encounter la sensitivity, khong phai primary filter (§4.2)

# ---------------- KHOA: data sufficiency gate (§4.4) ----------------
GATE_CELLS_PER_CLASS    = 500
GATE_SUBJECTS_PER_CLASS = 200
GATE_CELLS_TOTAL        = 1000
GATE_SUBJECTS_TOTAL     = 500
GATE_TOP1PCT_SHARE      = 0.15  # top 1% subject <= 15% contested initial gradient mass

# ---------------- KHOA: optimization ----------------
EPOCHS, EFF = 8, 256
LR_BLOCKS, LR_HEAD = 1e-5, 1e-5            # blocks 9-11 + backbone.LN | H5 norm + decoder
CLIP_BLOCKS, CLIP_HEAD = 5.0, 1.0
WD = 1e-2                                   # chi ndim>1; bias & LayerNorm wd=0
WARMUP_FRAC = 0.05
LAMBDA_SP = 0.0                             # §6.1: BO L2-SP, khong claim stabilization
HEAD_WARMUP_EPOCHS = 1                      # ep1: train blocks+backbone.LN, freeze H5 norm+mld

# ---------------- KHOA: ASL (y het nb11-15) ----------------
ASL_GP, ASL_GN, ASL_CLIP, ASL_EPS = 1, 4, 0.05, 1e-8

# ---------------- KHOA: eval ----------------
W_F, W_L   = 0.7, 0.3
MACRO_TOL  = 0.0015
N_BOOT     = 2000               # cluster = subject_id
IDENT_TOL  = 1e-6               # nguong identity gate 3 tang

PREREG = dict(
    plan='FINAL_PLAN_V3_3_LABEL_RELIABILITY.md',
    review='REVIEW_PLAN_V3_3_FINAL.md (5/5 chot) + REVIEW_PLAN_V3_2 §3-bis (loi CLS)',
    scope='PRE-GPU ONLY: 0a coverage+manifest, 0b gradient share+beta, 0c smoke, 0d calibration',
    arms=ARMS, eta=ETA, rare3=RARE3, worst3_fixed=WORST3_FIXED,
    beta_rule='beta_c = (eta*G_cont + G_ord)/(G_cont + G_ord), do tai INIT, logit-gradient level',
    claim_locked='gradient matched at the output/logit level at initialization',
    claim_forbidden='parameter-gradient matched throughout training',
    manifest=dict(window_days=WINDOW_DAYS, anchor='raw +1 only', nearest_anchor_per_cell=True,
                  same_view=REQUIRE_SAME_VIEW, same_encounter_primary=REQUIRE_SAME_ENCOUNTER,
                  adjacency_required=False, train_only=True),
    fracture_rule=('bidirectional trong primary; BAT BUOC bao cao D2 tach theo huong. '
                   'Neu gain chi tu future-positive anchor -> chi duoc goi "temporal association", '
                   'KHONG duoc claim sua omission. Backward-only la sensitivity, khong thay primary.'),
    sufficiency_gate=dict(cells_per_class=GATE_CELLS_PER_CLASS, subjects_per_class=GATE_SUBJECTS_PER_CLASS,
                          cells_total=GATE_CELLS_TOTAL, subjects_total=GATE_SUBJECTS_TOTAL,
                          top1pct_gradient_share=GATE_TOP1PCT_SHARE,
                          on_fail='R2 mat primary status; KHONG sua manifest roi tiep tuc'),
    loss_norm='L = sum(w*l) / (B_actual * C)  — HANG SO, khong chia sum(w)',
    token='patch-only: backbone.layernorm(h)[:, 1:]  (BO CLS)',
    tail='blocks 9-11 (0-indexed) trainable, ban sao rieng moi arm; blocks 0-8 frozen shared',
    layernorms='backbone.layernorm -> tail group, trainable tu ep1 | H5 norm -> head group, frozen ep1',
    l2sp=LAMBDA_SP, epochs=EPOCHS, seed=SEED, effective_batch=EFF,
    lr=dict(blocks=LR_BLOCKS, head=LR_HEAD), clip=dict(blocks=CLIP_BLOCKS, head=CLIP_HEAD), wd=WD,
    hard_failure_only=['NaN/Inf', 'shape sai', 'blocks 0-8 bi doi', 'blocks 9-11 khong update',
                       'optimizer/scaler hong', 'output tro thanh hang so do loi so hoc'],
    drift_policy='DIAGNOSTIC ONLY — khong dung de dung run hoac loai checkpoint (§7)',
    seed_staging=('seed 42 truoc de validate implementation; sau do chay 1337 va 2024 BAT KE dau '
                  'cua delta seed 42 (§11 — tranh selection bias)'),
    eta_limitation='D2 ~ 0 chi chung minh targeting khong huu ich TAI eta=0.10',
    eval='historical frozen labels la primary; known-only la diagnostic only')
json.dump(PREREG, open(f'{SC_DIR}/preregistration.json', 'w'), indent=1)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '|', torch.cuda.get_device_name(0) if dev == 'cuda' else 'CPU')
if dev == 'cuda': print(f'  VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
print(f'SEED {SEED} | arms {ARMS} | eta {ETA} | window +/-{WINDOW_DAYS}d | epochs {EPOCHS}')
print('PREREG khoa ->', f'{SC_DIR}/preregistration.json')
print('\nSCOPE: notebook nay KHONG train. Dung sau C9 (smoke) de doc so 0a/0b.')

## C0b · Stage ảnh từ shard vào ổ local

**VERBATIM từ nb13 cell 4.** Bước này nb16 thiếu ở bản đầu — `C1` assert `<1%` ảnh vắng nên nó
báo `346,578 anh thieu` khi `/content/cxr_local` còn rỗng.

Giải nén `cxr_560_shards/*.tar` từ Drive về `/content/cxr_local`. Có cờ `.done_<shard>` nên
chạy lại cell là **tiếp tục**, không giải nén lại từ đầu.

**Chi phí:** Drive đọc trần ~83 MB/s, tổng ~346k ảnh 560px. Lần đầu **~40–70 phút**, CPU/IO
thuần, không dùng GPU. Runtime Colab bị ngắt là phải stage lại (`/content` không bền) — nên chạy
cell này ngay sau `C0` rồi mới tính tới GPU.

In [ ]:
os.makedirs(LOCAL_IMG, exist_ok=True)
_have = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
if _have > 1000 and not os.path.isdir(SHARD_DIR):
    print(f'{_have:,} anh da co san o {LOCAL_IMG}; khong co SHARD_DIR -> bo qua staging')
else:
    tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar')) if os.path.isdir(SHARD_DIR) else []
    assert tars or _have > 1000, f'khong co shard o {SHARD_DIR} va {LOCAL_IMG} rong -> can data'
    todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
    print(f'shards: {len(tars)} | to stage: {len(todo)} | da co {_have:,} anh')
    for f in tqdm(todo, desc='staging shards', unit='shard'):
        tmp = f'/content/{f}'
        shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
        with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
        os.remove(tmp); open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()
    print('local images:', len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')]))

## C1 · Cohort + **nhãn thô bốn trạng thái**

Không dùng EHR nên C1 gọn hơn nb11–15 nhiều: chỉ cần nhãn, split, tier, view manifest.

Nhãn thô `{+1, 0, -1, blank}` **không còn** trong bất kứ file nào (01·C12 áp U-Ones rồi
`fillna(0)` **trước khi** ghi parquet). Phải query BigQuery một lần rồi cache ra **file mới**
`cohort_v2/cohort_raw_labels.parquet` — không ghi đè file cũ.

Assert bắt buộc: `U-Ones(raw) == nhãn cohort hiện có`, từng ô.

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; C = len(LABELS); assert C == 13
ehr = (pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'),
                       columns=['study_id', 'subject_id', 'split', 'setting'] + LABELS)
       .sort_values('study_id').reset_index(drop=True))
N = len(ehr)
Y_STUDY = ehr[LABELS].to_numpy('float32')
STUDY_ROW = {int(s): i for i, s in enumerate(ehr.study_id.to_numpy())}
SPLIT = ehr.split.to_numpy()
print(f'cohort {N:,} study | split {dict(zip(*np.unique(SPLIT, return_counts=True)))}')

# ---- tier + Rare-3 + NonTarget-10, KHOA tu class_freq.pkl (khong tinh lai) ----
cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
N_POS_TRAIN = np.asarray(cf['class_freq'], 'float64'); N_TRAIN = int(cf['num_train'])
TIER = np.where(N_POS_TRAIN/N_TRAIN < 0.02, 'tail',
                np.where(N_POS_TRAIN/N_TRAIN < 0.10, 'medium', 'head'))
RARE3_IDX = np.sort(np.argsort(N_POS_TRAIN, kind='stable')[:3])
assert set(LABELS[j] for j in RARE3_IDX) == set(RARE3), \
    f'Rare-3 lech: {[LABELS[j] for j in RARE3_IDX]} vs {RARE3}'
RARE3_IDX = np.array([LABELS.index(l) for l in RARE3])          # theo dung thu tu RARE3
NONTGT_IDX = np.setdiff1d(np.arange(C), RARE3_IDX)
W3_IDX = np.array([LABELS.index(l) for l in WORST3_FIXED])
TAIL_IDX = np.where(TIER == 'tail')[0]
print(f'Rare-3 (khoa): {[(l, int(N_POS_TRAIN[LABELS.index(l)])) for l in RARE3]}')
print(f'NonTarget-10 : {len(NONTGT_IDX)} nhan | Worst-3 fixed: {WORST3_FIXED}')

# ---- nhan tho 4 trang thai: cache ra FILE MOI ----
if os.path.exists(RAW_PQ):
    rawdf = pd.read_parquet(RAW_PQ); print(f'raw labels <- cache {RAW_PQ}')
else:
    from google.cloud import bigquery
    try:
        from google.colab import auth; auth.authenticate_user()
    except Exception:
        pass
    print('query BigQuery ...')
    chex = bigquery.Client(project=PROJECT, location='US').query(
        f'SELECT * FROM `{CXR}.chexpert`').to_dataframe()
    _nm = lambda s: re.sub(r'[^a-z]', '', str(s).lower())
    _col = {_nm(c): c for c in chex.columns}
    miss = [l for l in LABELS if _nm(l) not in _col]
    assert not miss, f'khong map duoc cot: {miss}'
    rawdf = chex.drop_duplicates('study_id').set_index('study_id')[[_col[_nm(l)] for l in LABELS]]
    rawdf.columns = [f'raw_{l}' for l in LABELS]
    rawdf = rawdf.reset_index()
    rawdf.to_parquet(RAW_PQ, index=False)
    print(f'raw labels -> {RAW_PQ}  ({len(rawdf):,} rows)  [FILE MOI, khong ghi de]')

RAW = (rawdf.drop_duplicates('study_id').set_index('study_id')
       .reindex(ehr.study_id.to_numpy())[[f'raw_{l}' for l in LABELS]].to_numpy('float64'))
assert RAW.shape == (N, C)

# ---- ASSERT: U-Ones(raw) phai tai tao dung nhan cohort ----
_y = np.where(np.isnan(RAW), 0.0, np.where(RAW == -1, 1.0, RAW)).astype('float32')
_bad = int((_y != Y_STUDY).sum())
print(f'\nU-Ones(raw) == cohort labels ? {_bad == 0}  ({_bad} o lech)')
assert _bad == 0, 'nhan tho KHONG tai tao duoc nhan cohort -> dung chay tiep'

# ---- state code: 1=+1, 0=explicit 0, -1=uncertain, 2=blank ----
STATE = np.where(np.isnan(RAW), 2, RAW).astype('int8')
IS_BLANK = STATE == 2
print(f'trang thai (toan cohort): +1 {int((STATE==1).sum()):>9,} | 0 {int((STATE==0).sum()):>9,} '
      f'| -1 {int((STATE==-1).sum()):>9,} | blank {int(IS_BLANK.sum()):>9,}')

# ---- view manifest (giong nb11-15) ----
vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal', 'lateral'])]
_dup = [c for c in ('subject_id', 'split', 'setting') if c in vm.columns]
man = vm.drop(columns=_dup).merge(ehr[['study_id', 'subject_id', 'split']], on='study_id', how='inner')
_before = len(man)
man = man[man.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))
          ].sort_values('dicom_id').reset_index(drop=True)
assert len(man) and (_before-len(man))/_before < 0.01, f'{_before-len(man)} anh thieu -> stage anh'
man['study_row'] = man.study_id.map(STUDY_ROW).astype('int64')
IDX = {s: np.where(man.split.to_numpy() == s)[0] for s in ('train', 'val', 'test')}
print(f'anh theo split: { {s: int(len(IDX[s])) for s in IDX} }')

# ---- view signature moi study: F / L / FL ----
# QUAN TRONG: tinh tu `vm` DAY DU, KHONG tu `man` (da loc theo file co san o /content).
# View signature la thuoc tinh cua study, khong phu thuoc vao anh nao da stage.
# Neu tinh tu `man`, study chua stage se co VSIG='' va '' == '' se KHOP SAI o C4.
VSIG = (vm.groupby('study_id')['view_group']
        .apply(lambda s: ''.join(sorted({x[0].upper() for x in s}))))
VSIG = VSIG.reindex(ehr.study_id.to_numpy()).fillna('').to_numpy()
print('view signature (tu view_map day du):', dict(zip(*np.unique(VSIG, return_counts=True))))
_nosig = int((VSIG == '').sum())
print(f'  study khong co view signature: {_nosig:,} -> KHONG BAO GIO duoc pair o C4')
assert _nosig / N < 0.02, f'{_nosig} study thieu view_group -> kiem tra view_map.parquet'
print('\nNOTE: TEST khong bao gio dung trong notebook nay.')

## C2 · Coverage audit + timestamp audit (**0a-1**)

In **số tuyệt đối** `+1 / 0 / -1 / blank` cho 13 nhãn × 3 split (§5.1 plan).

Cột `n(raw==0)` là chỗ đáng nhìn nhất: nó là lý do V2 (masking) bị chặn. V3.3 **không mask**
nên nó không còn chặn, nhưng vẫn cho biết `Endpoint K` sẽ nhỏ và lệch prevalence tới mức nào.

Timestamp audit (§5.4 → §12): đếm `missing`, `00:00:00`, `duplicate subject+time`. **Không**
giả định `00:00` luôn là date-only — chỉ đếm, rule quyết định sau khi thấy số.

In [ ]:
rows = []
for j, l in enumerate(LABELS):
    for sp in ('train', 'val', 'test'):
        m = SPLIT == sp
        v = STATE[m, j]
        n = int(m.sum())
        rows.append(dict(label=l, tier=TIER[j], rare3=l in RARE3, split=sp, n_study=n,
                         n_pos=int((v == 1).sum()), n_neg0=int((v == 0).sum()),
                         n_unc=int((v == -1).sum()), n_blank=int((v == 2).sum()),
                         known_rate=float(((v == 1) | (v == 0)).mean()),
                         prev_definite_pos=float((v == 1).mean()),
                         prev_definite_neg=float((v == 0).mean())))
CV = pd.DataFrame(rows)
CV.to_csv(f'{SC_DIR}/metrics/raw_label_counts.csv', index=False)
pd.set_option('display.width', 220)
print('=== TRAIN (§5.1) ===')
print(CV[CV.split == 'train'].drop(columns=['split', 'n_study']).to_string(
      index=False, float_format=lambda v: f'{v:.4f}'))

print('\n=== n(raw==0) — known negative, theo split ===')
print(CV.pivot_table(index='label', columns='split', values='n_neg0')
        .reindex(LABELS)[['train', 'val', 'test']].to_string())
_r3 = CV[(CV.split == 'train') & CV.rare3]
print('\nRare-3 TRAIN known negative:',
      {r.label: int(r.n_neg0) for r in _r3.itertuples()})
print('  (V2 bi chan vi so nay < 1000; V3.3 KHONG mask nen khong con chan.')
print('   Van ghi lai vi no quyet dinh co mau va prevalence cua Endpoint K.)')

# ---------------- timestamp audit ----------------
_base = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_base.parquet'),
                        columns=['study_id', 'subject_id', 'index_time'])
_base = _base.drop_duplicates('study_id').set_index('study_id').reindex(ehr.study_id.to_numpy())
T = pd.to_datetime(_base['index_time'])
TSEC = T.astype('int64').to_numpy() / 1e9                       # giay, NaT -> rat am
n_missing = int(T.isna().sum())
_midnight = ((T.dt.hour == 0) & (T.dt.minute == 0) & (T.dt.second == 0)).fillna(False)
_dupkey = pd.Series(ehr.subject_id.astype(str).to_numpy() + '|' + T.astype(str).to_numpy())
n_dup = int(_dupkey.duplicated(keep=False).sum())
TS_OK = (~T.isna()).to_numpy()
print(f'\n=== timestamp audit (§5.4) ===')
print(f'  missing index_time        {n_missing:>8,}  ({n_missing/N:.3%})')
print(f'  time == 00:00:00          {int(_midnight.sum()):>8,}  ({_midnight.mean():.3%})')
print(f'  duplicate subject+time    {n_dup:>8,}  ({n_dup/N:.3%})')
json.dump(dict(n_study=N, missing_index_time=n_missing, midnight=int(_midnight.sum()),
               duplicate_subject_time=n_dup, rule='CHUA CHOT — quyet dinh sau khi doc so nay'),
          open(f'{SC_DIR}/metrics/timestamp_audit.json', 'w'), indent=1)
print(f'''
  RULE CHUA CHOT (§12 plan). Doc so tren roi quyet dinh:
    - midnight < ~1%  -> loai chung khoi manifest (an toan, mat it cell)
    - midnight nhieu  -> GIU, vi 00:00 co the la gio chup that; ghi nhan la nguon nhieu
  Hien tai TS_OK = (index_time khong NaT). Doi lai o C4 neu muon chat hon.''')

## C3 · Encounter metadata

`cur_hadm` / `cur_edstay` nằm ở **`cohort_base.parquet`** (01·C18 `base_cols`), **không** ở
`cohort_study_level.parquet` mà nb11–15 đang load. Phải merge thêm.

ICU `stay_id` **không có** trong cohort ⇒ điều kiện ICU của các bản plan cũ rơi về `hadm_id`.

Encounter **không** phải primary filter (§4.2) — chỉ dùng để báo cáo sensitivity.

In [ ]:
_bcols = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_base.parquet')).columns.tolist()
print('cot co trong cohort_base.parquet:', _bcols)
_want = [c for c in ('cur_hadm', 'cur_edstay') if c in _bcols]
assert _want, 'khong thay cur_hadm/cur_edstay -> khong the bao cao sensitivity theo encounter'
enc = (pd.read_parquet(os.path.join(OUT_DIR, 'cohort_base.parquet'),
                       columns=['study_id'] + _want)
       .drop_duplicates('study_id').set_index('study_id').reindex(ehr.study_id.to_numpy()))
HADM = enc['cur_hadm'].to_numpy() if 'cur_hadm' in enc else np.full(N, np.nan)
EDST = enc['cur_edstay'].to_numpy() if 'cur_edstay' in enc else np.full(N, np.nan)
print(f'  cur_hadm   co gia tri {np.sum(pd.notna(HADM)):>9,} / {N:,}  ({pd.notna(HADM).mean():.1%})')
print(f'  cur_edstay co gia tri {np.sum(pd.notna(EDST)):>9,} / {N:,}  ({pd.notna(EDST).mean():.1%})')
print('  ICU stay_id: KHONG co trong cohort -> khong dung')

## C4 · Temporal manifest ±7 ngày + sufficiency gate (**0a-2**)

Một ô `(study, label)` là **contested** khi tồn tại anchor thoả (§4.1 plan):

```
cung subject_id
raw cua chinh study  = blank
raw cua anchor       = +1        (KHONG dung raw -1)
0 < |delta_t| <= 7 ngay
cung view signature
timestamp dung/tin cay ca hai ben
```

**Không** yêu cầu chronological-adjacent — quét **mọi** study của bệnh nhân trong cửa sổ, rồi
lấy **anchor gần nhất** cho mỗi ô blank, đánh dấu **một lần**. Nhờ vậy khai thác hết bằng chứng
dọc thời gian mà không tạo mẫu bậc hai.

Chỉ dùng **TRAIN** để tạo weight. Cửa sổ 7 ngày là **persistence prior** cho tổn thương cấu
trúc, không phải bằng chứng hai study có cùng trạng thái bệnh.

### `Fracture` — quy tắc khoá riêng

Giữ **hai chiều** trong primary, nhưng bắt buộc báo cáo `D2` tách theo hướng:

| hướng | diễn giải cho phép |
|---|---|
| anchor **trước** blank | persistence/omission — tương đối mạnh |
| blank → anchor **sau** | có thể là gãy **mới** — yếu hơn |

Nếu gain của `Fracture` **chỉ** đến từ future-positive anchor: chỉ được gọi **temporal
association**, **không** được claim sửa omission.

In [ ]:
IS_TRAIN = SPLIT == 'train'
SUBJ = ehr.subject_id.to_numpy()
DAY = 86400.0
WIN = WINDOW_DAYS * DAY

CONTESTED = np.zeros((N, C), bool)                 # o (study, label) bi tranh chap
GAP_H     = np.full((N, C), np.nan, 'float32')     # gio, tri tuyet doi
ANCH_ROW  = np.full((N, C), -1, 'int64')           # study row cua anchor gan nhat
FUNNEL, DETAIL = [], []

_ord = np.argsort(SUBJ, kind='stable')
_bnd = np.r_[0, np.flatnonzero(np.diff(SUBJ[_ord])) + 1, len(_ord)]
_groups = [_ord[_bnd[k]:_bnd[k+1]] for k in range(len(_bnd)-1)]
print(f'{len(_groups):,} subject | quet cua so +/-{WINDOW_DAYS}d cho {len(RARE3)} nhan Rare-3\n')

for l in RARE3:
    j = LABELS.index(l)
    n_cand = n_ts = n_win = n_view = 0
    for g in _groups:
        g = g[IS_TRAIN[g]]
        if len(g) < 2: continue
        st = STATE[g, j]
        ip, ib = g[st == 1], g[st == 2]
        if not len(ip) or not len(ib): continue
        n_cand += len(ib)
        okp, okb = TS_OK[ip], TS_OK[ib]
        ip, ib = ip[okp], ib[okb]
        if not len(ip) or not len(ib): continue
        n_ts += len(ib)
        d = np.abs(TSEC[ib][:, None] - TSEC[ip][None, :])          # [n_blank, n_pos]
        ok = (d > 0) & (d <= WIN)
        n_win += int(ok.any(1).sum())
        if REQUIRE_SAME_VIEW:
            _vb, _vp = VSIG[ib], VSIG[ip]
            ok &= (_vb[:, None] == _vp[None, :]) & (_vb[:, None] != '') & (_vp[None, :] != '')
        hit = ok.any(1)
        n_view += int(hit.sum())
        if not hit.any(): continue
        dm = np.where(ok, d, np.inf)
        k = dm.argmin(1)
        for bi, ki, h in zip(ib[hit], k[hit], hit.nonzero()[0]):
            a = ip[ki]; gap = float(dm[h, ki])
            CONTESTED[bi, j] = True; GAP_H[bi, j] = gap/3600.0; ANCH_ROW[bi, j] = a
            DETAIL.append((l, int(SUBJ[bi]), int(bi), int(a), gap/3600.0,
                           'past' if TSEC[a] < TSEC[bi] else 'future',
                           bool(pd.notna(HADM[bi]) and HADM[bi] == HADM[a]),
                           bool(pd.notna(EDST[bi]) and EDST[bi] == EDST[a])))
    n_fin = int(CONTESTED[:, j].sum())
    FUNNEL.append(dict(label=l, blank_with_pos_subject=n_cand, plus_timestamp_ok=n_ts,
                       plus_within_window=n_win, plus_same_view=n_view, final=n_fin,
                       n_blank_train=int((IS_BLANK[:, j] & IS_TRAIN).sum()),
                       frac_of_blank=n_fin/max(int((IS_BLANK[:, j] & IS_TRAIN).sum()), 1)))
FUN = pd.DataFrame(FUNNEL); FUN.to_csv(f'{SC_DIR}/metrics/temporal_manifest_funnel.csv', index=False)
print('=== FUNNEL (§5.2) ===')
print(FUN.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

DET = pd.DataFrame(DETAIL, columns=['label','subject_id','study_row','anchor_row','gap_h',
                                    'direction','same_hadm','same_edstay'])
DET['gap_bin'] = pd.cut(DET.gap_h, [0, 24, 72, 24*7],
                        labels=['<=24h', '24-72h', '72h-7d'], include_lowest=True)
DET.to_parquet(f'{SC_DIR}/metrics/contested_cells.parquet', index=False)

print('\n=== CONCENTRATION + huong + gap (§4.3, §5.5) ===')
crows = []
for l in RARE3:
    d = DET[DET.label == l]
    per = d.groupby('subject_id').size().sort_values(ascending=False)
    top1 = per.iloc[:max(1, int(np.ceil(0.01*len(per))))].sum() / max(len(d), 1)
    crows.append(dict(label=l, cells=len(d), studies=d.study_row.nunique(),
                      subjects=len(per), median_cells_per_subject=float(per.median()) if len(per) else 0.0,
                      max_cells_one_subject=int(per.max()) if len(per) else 0,
                      top1pct_CELL_share=float(top1),
                      past=int((d.direction == 'past').sum()), future=int((d.direction == 'future').sum()),
                      **{f'gap_{b}': int((d.gap_bin == b).sum()) for b in ['<=24h','24-72h','72h-7d']},
                      same_hadm=int(d.same_hadm.sum()), same_edstay=int(d.same_edstay.sum())))
CON = pd.DataFrame(crows); CON.to_csv(f'{SC_DIR}/metrics/contested_cells_by_class.csv', index=False)
print(CON.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print('\n  LUU Y: top1pct_CELL_share la ty phan theo SO CELL. Gate §4.4 doi ty phan theo')
print('  GRADIENT MASS -> tinh o C8 sau khi co g_{k,c}. Con so o day chi de xem truoc.')

print('\n=== DATA SUFFICIENCY GATE (§4.4) — GRADIENT SHARE se kiem lai o C8 ===')
_tc, _ts_ = int(CON.cells.sum()), DET.subject_id.nunique()
_pass_cls = {r.label: dict(cells=r.cells >= GATE_CELLS_PER_CLASS,
                           subjects=r.subjects >= GATE_SUBJECTS_PER_CLASS) for r in CON.itertuples()}
for l, v in _pass_cls.items():
    r = CON[CON.label == l].iloc[0]
    print(f'  {l:<16} cells {int(r.cells):>6,} (>={GATE_CELLS_PER_CLASS}) {"OK" if v["cells"] else "FAIL":<4} | '
          f'subjects {int(r.subjects):>6,} (>={GATE_SUBJECTS_PER_CLASS}) {"OK" if v["subjects"] else "FAIL"}')
print(f'  Rare-3 TONG      cells {_tc:>6,} (>={GATE_CELLS_TOTAL}) '
      f'{"OK" if _tc >= GATE_CELLS_TOTAL else "FAIL":<4} | subjects {_ts_:>6,} '
      f'(>={GATE_SUBJECTS_TOTAL}) {"OK" if _ts_ >= GATE_SUBJECTS_TOTAL else "FAIL"}')
_gate_ok = all(all(v.values()) for v in _pass_cls.values()) and _tc >= GATE_CELLS_TOTAL and _ts_ >= GATE_SUBJECTS_TOTAL
print(f'''
  KET LUAN TAM ({"PASS" if _gate_ok else "FAIL"}) — con cho gradient-share o C8.
  Neu FAIL: theo §4.4, R2 MAT primary status. KHONG duoc sua manifest roi tiep tuc claim
  cung estimand. Sensitivity "cap <= 3 cells/subject" duoc phep nhung phai ghi exploratory.''')
np.save(f'{SC_DIR}/metrics/contested_mask.npy', CONTESTED)
json.dump(dict(gate_cells_subjects_pass=bool(_gate_ok), total_cells=_tc, total_subjects=int(_ts_)),
          open(f'{SC_DIR}/metrics/sufficiency_partial.json', 'w'), indent=1)

## C5 · Dataset + loader — **không EHR**

nb16 không dùng EHR (§1 plan) nên dataset gọn hơn nb11–15 rất nhiều: chỉ `image`, `y`,
`study_row`. Không `EmbeddingBag`, không ragged bag, không `set_id/rec_id/stat_id`.

`study_row` cần cho weight matrix — mỗi ảnh thừa hưởng weight của study nó thuộc về.

`num_workers=0` cho mọi eval/predict loader (memory: `dataloader-teardown-spam`).

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T
from PIL import Image
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(MODEL_SIZE, scale=(0.85,1.0), ratio=(0.95,1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
eval_tf  = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
print(f'MODEL_SIZE {MODEL_SIZE} | n_patch = {(MODEL_SIZE//14)**2} | +CLS = {(MODEL_SIZE//14)**2+1}')

class ImgDS(Dataset):
    """Anh + nhan study + study_row. KHONG co EHR."""
    def __init__(s, idx, tf): s.idx, s.tf = np.asarray(idx), tf
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        r = man.iloc[s.idx[k]]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        im = Image.merge('RGB', (im, im, im))
        tr = int(r.study_row)
        return dict(image=s.tf(im), y=torch.from_numpy(Y_STUDY[tr]), sr=tr)

def collate(bb):
    return dict(image=torch.stack([x['image'] for x in bb]),
                y=torch.stack([x['y'] for x in bb]),
                sr=torch.tensor([x['sr'] for x in bb], dtype=torch.int64))

def loader(split, tf, shuffle, bs, workers=8):
    return DataLoader(ImgDS(IDX[split], tf), batch_size=bs, shuffle=shuffle, collate_fn=collate,
                      num_workers=workers, pin_memory=(dev == 'cuda'), persistent_workers=(workers > 0))

to_dev = lambda b: {k: v.to(dev, non_blocking=True) for k, v in b.items()}
_b = to_dev(collate([ImgDS(IDX['val'], eval_tf)[k] for k in range(4)]))
print('batch keys:', list(_b), '| image', tuple(_b['image'].shape), '| y', tuple(_b['y'].shape))
assert (Y_STUDY[_b['sr'].cpu().numpy()] == _b['y'].cpu().numpy()).all(), 'sr khong khop y'
print('dataset OK (khong EHR)')

## C6 · Model — blocks 0–8 frozen shared · **9–11 trainable bản sao riêng** · H5 head patch-only

Ba nhóm tham số, khác hoàn toàn nb11–15:

| nhóm | module | thuộc | lr | clip | ep1 |
|---|---|---|---:|---:|---|
| shared frozen | `backbone.embeddings` + blocks 0–8 | dùng chung, `no_grad` | — | — | frozen |
| **tail** | blocks 9–11 + `backbone.layernorm` | **bản sao riêng mỗi arm** | `1e-5` | 5.0 | **train** |
| head | H5 `norm` + `mld` | bản sao riêng mỗi arm | `1e-5` | 1.0 | **frozen** |

`backbone.layernorm` thuộc nhóm **tail** (§5.2) — nó là LayerNorm cuối của backbone, không phải
của head. H5 `norm` là LayerNorm riêng của head, đóng băng ở ep1 cùng decoder.

### Patch-only — sửa lỗi CLS của nb14/15

```python
h = tail(z_shared)              # blocks 9,10,11 -> backbone.layernorm
logits = head(h[:, 1:])         # BO CLS  <-- nb14/15 thieu dong nay
```

Vì blocks 9–11 giờ **trainable** nên `forward_shared` chỉ chạy tới block 8 — rẻ hơn nb13
(chạy 0–8 no_grad) nhưng phần đuôi đắt hơn nhiều vì phải giữ activation cho backward qua 3
block × 1,370 token.

In [ ]:
from transformers import AutoModel

TRAIN_FROM = 9              # 0-indexed: blocks 9,10,11 trainable (= block 10,11,12 1-indexed)
_probe = AutoModel.from_pretrained('microsoft/rad-dino')
N_BLOCKS = len(_probe.encoder.layer); D = _probe.config.hidden_size; del _probe
assert D == 768 and TRAIN_FROM < N_BLOCKS
N_PATCH = (MODEL_SIZE // 14) ** 2

assert os.path.exists(BB_CKPT), f'04b backbone khong thay: {BB_CKPT}'
_sd04b = torch.load(BB_CKPT, map_location='cpu', weights_only=False)['model']
BB_SD = {k[9:]: v for k, v in _sd04b.items() if k.startswith('backbone.')}
assert BB_SD, 'khong co backbone.* trong 04b checkpoint'

def load_backbone(device=dev):
    m = AutoModel.from_pretrained('microsoft/rad-dino')
    m.load_state_dict(BB_SD, strict=True)
    return m.to(device).eval()

# ---------------- shared frozen: embeddings + blocks 0..TRAIN_FROM-1 ----------------
backbone = load_backbone()
for p in backbone.parameters(): p.requires_grad_(False)
LAYERS = backbone.encoder.layer
def _layer(h, i):
    o = LAYERS[i](h); return o[0] if isinstance(o, (tuple, list)) else o

@torch.no_grad()
def forward_shared(x):
    h = backbone.embeddings(x)
    for i in range(TRAIN_FROM): h = _layer(h, i)
    return h.detach()

# ---------------- H5 head ----------------
import sys, types
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder
if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched'):
    _oi = _mld.TransformerDecoderLayerOptimal.__init__
    def _init(self, *a, **kw): _oi(self, *a, **kw); self.self_attn = types.SimpleNamespace(batch_first=False)
    _mld.TransformerDecoderLayerOptimal.__init__ = _init
    _of = _mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self, tgt, memory, **kw):
        kw.pop('tgt_is_causal', None); kw.pop('memory_is_causal', None); return _of(self, tgt, memory, **kw)
    _mld.TransformerDecoderLayerOptimal.forward = _fwd
    _mld.TransformerDecoderLayerOptimal._patched = True

assert os.path.exists(H5_CKPT), f'H5 head khong thay: {H5_CKPT}'
H5_SD = torch.load(H5_CKPT, map_location='cpu', weights_only=False)['model']
_sub = lambda p: {k[len(p):]: v for k, v in H5_SD.items() if k.startswith(p)}
assert _sub('norm.') and _sub('mld.'), f'H5 thieu norm.*/mld.*: {list(H5_SD)[:5]}'
print(f'H5 head: norm {len(_sub("norm."))} tensor | mld {len(_sub("mld."))} tensor')

# ---------------- per-arm modules ----------------
_TAIL_SRC = load_backbone('cpu')                # nguon deepcopy, tao 1 lan
for p in _TAIL_SRC.parameters(): p.requires_grad_(False)

class TailBackbone(nn.Module):
    """Blocks TRAIN_FROM..11 + backbone.layernorm — TRAINABLE, ban sao rieng moi arm."""
    def __init__(s):
        super().__init__()
        s.blocks = nn.ModuleList([copy.deepcopy(_TAIL_SRC.encoder.layer[i])
                                  for i in range(TRAIN_FROM, N_BLOCKS)])
        s.ln = copy.deepcopy(_TAIL_SRC.layernorm)
        for p in s.parameters(): p.requires_grad_(True)
    def forward(s, h):
        for b in s.blocks:
            o = b(h); h = o[0] if isinstance(o, (tuple, list)) else o
        return s.ln(h)

class H5Head(nn.Module):
    """H5 norm + ML-Decoder, patch-only input."""
    def __init__(s):
        super().__init__()
        s.norm = nn.LayerNorm(D)
        s.mld = MLDecoder(num_classes=C, initial_num_features=D, decoder_embedding=D, num_of_groups=C)
        s.norm.load_state_dict(_sub('norm.')); s.mld.load_state_dict(_sub('mld.'))
    def forward(s, patches): return s.mld(s.norm(patches))

class Arm(nn.Module):
    def __init__(s):
        super().__init__(); s.tail = TailBackbone(); s.head = H5Head()
    def forward(s, z_shared):
        h = s.tail(z_shared)
        assert h.shape[1] == N_PATCH + 1, f'token {h.shape[1]} != {N_PATCH+1}'
        return s.head(h[:, 1:])                      # <-- PATCH-ONLY, bo CLS

def build_arm(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    return Arm().to(dev)

tail_params = lambda m: list(m.tail.parameters())
head_params = lambda m: list(m.head.parameters())

_A = build_arm()
_np = lambda ps: sum(p.numel() for p in ps)
print(f'\ntrainable/arm {_np(tail_params(_A))+_np(head_params(_A)):>12,}'
      f'  = tail {_np(tail_params(_A)):,} + head {_np(head_params(_A)):,}')
print(f'frozen shared  {_np(list(backbone.parameters())):>12,}  (dung chung {len(ARMS)} arm)')
print(f'snapshot/arm   ~{(_np(tail_params(_A))+_np(head_params(_A)))*4/1e6:.0f} MB')

# ---- G0: forward_shared + tail == backbone(x) stock (tai init) ----
_x = torch.randn(2, 3, MODEL_SIZE, MODEL_SIZE, device=dev)
with torch.no_grad():
    _ref = backbone(pixel_values=_x).last_hidden_state
    _got = _A.tail(forward_shared(_x))
_d = (_ref - _got).abs().max().item()
print(f'\nG0 forward_shared+tail == backbone(x).last_hidden_state  max|diff| {_d:.2e} '
      f'{"OK" if _d < 1e-4 else "FAIL"}')
assert _d < 1e-4
print(f'   -> ban sao blocks {TRAIN_FROM}..{N_BLOCKS-1} + layernorm giu dung trong so 04b')

## C7 · **Identity gate ba tầng** vs `ArmModel(plain=True)` của nb13 + stochasticity

Chốt ở `REVIEW_PLAN_V3_3_FINAL.md` §2·Q1. Năm điều kiện:

1. reference khởi tạo **trực tiếp từ code/class của nb13** — không viết lại `forward` cho nb16;
2. load backbone 04b và H5 `norm`/`mld` **độc lập** (instance riêng);
3. reference và candidate **không dùng chung** object/module nào;
4. `eval()`, FP32, cùng một batch, không augmentation;
5. so **ba tầng**, không chỉ logit.

| tầng | so cái gì | vì sao |
|---|---|---|
| **L1** | token sau **block 11**, *trước* final LN | lệch ở đây ⇒ lỗi ở bản sao block hoặc thứ tự chạy |
| **L2** | `backbone.layernorm(h)[:, 1:]` | lệch chỉ ở đây ⇒ lỗi ở final LN hoặc **chỗ cắt CLS** |
| **L3** | logits sau H5 head | lệch chỉ ở đây ⇒ lỗi ở `norm`/`mld` |

Ngưỡng `max|diff| ≤ 1e-6`. Nếu chỉ so L3 thì khi fail chỉ biết "có lệch", không biết ở đâu —
đó là lý do dùng ba tầng.

**Cổng `G2` của nb14 không bắt được lỗi CLS** vì nó so hai instance của *cùng* một class. Ở đây
reference đọc `h[:, 1:]` trong forward của **nb13**, candidate đọc `h[:, 1:]` trong forward của
nb16 — hai code path riêng. Nếu một bên quên `[:, 1:]` thì L2 và L3 sẽ khác.

Lấy L1/L2 bằng **forward hook** để `_r_forward_tail` giữ nguyên verbatim, không phải sửa nó.

### Stochasticity (§7 plan · 0c)

RAD-DINO/Dinov2 mặc định `dropout = 0` và `drop_path_rate = 0`. Cần **in ra** thay vì giả định:
nếu bằng 0 thì `train()` và `eval()` cho cùng output ⇒ **không** cần `sync_rng`, và mọi khác
biệt giữa arm chỉ đến từ weight.

In [ ]:
# ================== REFERENCE: VERBATIM tu 13_dbl_weight_screen.ipynb ==================
# Instance backbone RIENG, khong dung chung voi `backbone` cua nb16.
_rb = load_backbone()
for p in _rb.parameters(): p.requires_grad_(False)
_rLAYERS = _rb.encoder.layer; _rn_blocks = len(_rLAYERS)
_rLATE = (9, 10)                              # nb13: LATE_1IDX=(10,11) -> 0-indexed (9,10)
_rSHARED_UPTO = min(_rLATE)                   # = 9, trung TRAIN_FROM
assert _rSHARED_UPTO == TRAIN_FROM

def _r_layer(h, i):                                                   # verbatim nb13
    o = _rLAYERS[i](h); return o[0] if isinstance(o, (tuple, list)) else o

def _r_forward_shared(x):                                             # verbatim nb13
    with torch.no_grad():
        h = _rb.embeddings(x)
        for i in range(_rSHARED_UPTO): h = _r_layer(h, i)
    return h.detach()

def _r_forward_tail(h, adapter_blocks=(), adapters=None, e=None):     # verbatim nb13
    first = min(adapter_blocks) if adapter_blocks else _rn_blocks
    for i in range(_rSHARED_UPTO, _rn_blocks):
        with (torch.no_grad() if i < first else contextlib.nullcontext()):
            h = _r_layer(h, i)
        if adapters is not None and str(i) in adapters:
            h = adapters[str(i)](h, e)
    return _rb.layernorm(h)

class _RefArmModel(nn.Module):                                        # verbatim nb13 (nhanh plain)
    def __init__(self, plain=False):
        super().__init__(); self.plain = plain; self.blocks = () if plain else tuple(_rLATE)
        self.norm = nn.LayerNorm(768); self.mld = MLDecoder(num_classes=len(LABELS),
                     initial_num_features=768, decoder_embedding=768, num_of_groups=len(LABELS))
        self.norm.load_state_dict(_sub('norm.')); self.mld.load_state_dict(_sub('mld.'))
    def forward(self, z_shared, batch=None):
        if self.plain: h = _r_forward_tail(z_shared)
        else: raise NotImplementedError('nb16 chi dung nhanh plain')
        return self.mld(self.norm(h[:, 1:]))

REF = _RefArmModel(plain=True).to(dev).eval()
for p in REF.parameters(): p.requires_grad_(False)

# --- dieu kien 3: khong dung chung module ---
_id_ref = {id(m) for m in list(REF.modules()) + list(_rb.modules())}
_id_cnd = {id(m) for m in list(_A.modules()) + list(backbone.modules())}
_shared_mod = _id_ref & _id_cnd
print(f'module dung chung REF vs CANDIDATE: {len(_shared_mod)} '
      f'{"OK (doc lap hoan toan)" if not _shared_mod else "FAIL"}')
assert not _shared_mod, 'reference va candidate dung chung module -> khong phai code path doc lap'

# ================== ba tang, qua forward hook ==================
_cap = {}
def _hook(tag): return lambda mod, inp, out: _cap.__setitem__(
    tag, (out[0] if isinstance(out, (tuple, list)) else out).detach().float())
_hs = [_rLAYERS[_rn_blocks-1].register_forward_hook(_hook('r_L1')),
       _rb.layernorm.register_forward_hook(_hook('r_L2')),
       _A.tail.blocks[-1].register_forward_hook(_hook('c_L1')),
       _A.tail.ln.register_forward_hook(_hook('c_L2'))]

_A.eval(); REF.eval()
_bat = to_dev(collate([ImgDS(IDX['val'], eval_tf)[k] for k in range(8)]))   # eval_tf: khong aug
with torch.no_grad():
    _zr = _r_forward_shared(_bat['image']);  _lr = REF(_zr, None)           # code path nb13
    _zc = forward_shared(_bat['image']);     _lc = _A(_zc)                  # code path nb16
for h in _hs: h.remove()

_L = [('L0  shared: token sau block 8',      _zr,                 _zc),
      ('L1  sau block 11 (truoc final LN)',  _cap['r_L1'],        _cap['c_L1']),
      ('L2  layernorm(h)[:, 1:] patch-only', _cap['r_L2'][:, 1:], _cap['c_L2'][:, 1:]),
      ('L3  logits sau H5 head',             _lr,                 _lc)]
print(f'\n=== IDENTITY GATE BA TANG (nguong {IDENT_TOL:.0e}) ===')
IDENT = {}
for nm, a, b in _L:
    a, b = a.float(), b.float()
    assert a.shape == b.shape, f'{nm}: shape {tuple(a.shape)} vs {tuple(b.shape)}'
    d = (a - b).abs().max().item()
    IDENT[nm.split()[0]] = d
    print(f'  {nm:<38} shape {str(tuple(a.shape)):<20} max|diff| {d:.3e}  '
          f'{"OK" if d <= IDENT_TOL else "FAIL"}')
assert all(v <= IDENT_TOL for v in IDENT.values()), (
    'identity gate FAIL — doc tang dau tien lech de dinh vi loi:\n'
    '  L1 lech -> ban sao block 9-11 hoac thu tu chay\n'
    '  chi L2 lech -> final LayerNorm hoac cho cat CLS\n'
    '  chi L3 lech -> H5 norm/mld')

# --- kiem chung nguoc: CO TINH giu CLS -> L2/L3 PHAI lech ---
with torch.no_grad():
    _bad = _A.head(_cap['c_L2'])                      # giu ca CLS = dung loi cua nb14/15
_dbad = (_bad - _lc).abs().max().item()
print(f'\n  kiem chung nguoc: neu GIU CLS thi logits lech {_dbad:.3e} '
      f'{"OK (cong nay co kha nang bat loi)" if _dbad > 1e-4 else "!! cong VO DUNG"}')
assert _dbad > 1e-4, 'giu CLS ma logits khong doi -> cong nay khong bat duoc loi gi'
print(f'  -> day chinh la loi cua nb14/15 (REVIEW_PLAN_V3_2 §3-bis). nb16 khong con.')
json.dump(dict(tol=IDENT_TOL, levels=IDENT, cls_counterfactual_diff=_dbad,
               reference='ArmModel(plain=True) verbatim tu 13_dbl_weight_screen.ipynb cell 14',
               shared_modules=0), open(f'{SC_DIR}/metrics/identity_gate.json', 'w'), indent=1)

# ================== stochasticity (§7 · 0c) ==================
cfg = backbone.config
_g = lambda k: getattr(cfg, k, None)
print(f'\n=== STOCHASTICITY ===')
for k in ('hidden_dropout_prob', 'attention_probs_dropout_prob', 'drop_path_rate',
          'layerscale_value', 'drop_path_uniform'):
    print(f'  cfg.{k:<30} {_g(k)}')
_A.train()
with torch.no_grad():
    _o1 = _A(_zc); _o2 = _A(_zc)
_A.eval()
with torch.no_grad(): _o3 = _A(_zc)
_dtt = (_o1 - _o2).abs().max().item(); _dte = (_o1 - _o3).abs().max().item()
print(f'  train() 2 lan       max|diff| {_dtt:.3e}')
print(f'  train() vs eval()   max|diff| {_dte:.3e}')
STOCHASTIC = _dtt > 1e-7
print(f'''
  KET LUAN: forward {"CO" if STOCHASTIC else "KHONG CO"} thanh phan ngau nhien.
  {"-> PHAI dung sync_rng(ep, i) tung arm tung step de dropout mask trung nhau." if STOCHASTIC else
   "-> KHONG can sync_rng. Moi khac biet giua arm chi den tu weight (do weight matrix)."}
  {"" if STOCHASTIC else "  (dropout=0 va drop_path=0 la mac dinh cua Dinov2/RAD-DINO — da xac nhan bang so.)"}''')
json.dump(dict(stochastic=bool(STOCHASTIC), train_vs_train=_dtt, train_vs_eval=_dte,
               cfg={k: _g(k) for k in ('hidden_dropout_prob','attention_probs_dropout_prob','drop_path_rate')}),
          open(f'{SC_DIR}/metrics/stochasticity.json', 'w'), indent=1)
del REF, _rb, _cap
torch.cuda.empty_cache() if dev == 'cuda' else None

## C8 · Gradient share → `β_c` (**0b** + **0d**) — hiệu chuẩn trên **phân phối train**

Với `y = 0`, số hạng âm của ASL là

$$\ell^-(z)=-\ln(x_n)\,(1-x_n)^{\gamma_n},\qquad x_n=\min(1-\sigma(z)+0.05,\;1)$$

và $g=\left|\partial\ell^-/\partial z\right|$, tính bằng **autograd trên đúng hàm loss đang
dùng** rồi đối chiếu dạng đóng.

$$\beta_c=\frac{\eta G_{\text{cont},c}+G_{\text{ord},c}}{G_{\text{cont},c}+G_{\text{ord},c}}
=1-(1-\eta)\,s_c$$

`NonTarget-10` có `G_cont = 0` ⇒ `β = 1` tự động; `β_c ∈ [η, 1]` luôn.

### `β_train` là primary, `β_eval` là sensitivity *(chốt 2026-07-30)*

Bản đầu tôi hiệu chuẩn `β` trên `eval_tf` (CenterCrop, không tăng cường) rồi **áp lên** training
chạy `train_tf` (RandomResizedCrop 0.85–1.0 + rotate 7°). Hai phân phối khác nhau.

Điều đó không phải chuyện nhỏ: `Fracture` và `Pleural_Other` là tổn thương **ngoại vi** (xương
sườn, rìa màng phổi). Crop xuống 85% có thể cắt mất chúng **thường xuyên hơn** tổn thương trung
tâm ⇒ tăng cường tác động lên ô contested **khác** ô blank thường, và `β` đo trên center crop
không còn khớp trên phân phối train thực.

Nên: **primary = `β_train`**, gộp từ **hai augmentation seed cố định** `(42, 1337)`. Hai seed cho
hai việc — pooled mass giảm nhiễu, và **hiệu số `β` giữa hai seed** cho biết bản thân `β` nhạy
thế nào với ngẫu nhiên tăng cường. Nếu hiệu số đó lớn hơn khoảng cách `β_train` ↔ `β_eval` thì
tranh luận eval-vs-train là vô nghĩa ngay từ đầu.

Tính hợp lệ prereg: `β` **chưa** bị khoá. Plan §13 đặt lock ở bước 1, **sau** `0a`–`0d`. Đây
đúng là bước để quyết phân phối hiệu chuẩn rồi mới khoá.

### Tăng cường phải **tái lập được**

`β` sẽ vào prereg nên hai lượt `train_tf` phải cho đúng cùng một chuỗi crop mỗi lần chạy.
Dùng `worker_init_fn` gieo từng worker + `shuffle=False`, rồi **kiểm bằng số**: chạy 3 batch đầu
hai lần và assert bằng nhau bit-for-bit; đồng thời assert hai seed khác nhau **có** cho ảnh khác
nhau. Nếu kiểm thất bại thì tự hạ về `workers=0` (tất định hiển nhiên) rồi kiểm lại.

Lập luận suông "worker seeding là tất định" không đủ — nó phụ thuộc số worker, phiên bản
torch, và `persistent_workers`. Kiểm bằng số thì không.

### Chi phí

| lượt | transform | cache |
|---|---|---|
| `eval_tf` | CenterCrop | đã có từ lần chạy trước — **miễn phí** |
| `train_tf` seed 42 | tăng cường | ~15–35 phút |
| `train_tf` seed 1337 | tăng cường | ~15–35 phút |

Mỗi lượt ghi file riêng, chạy lại là đọc cache. File `initial_gradient_share.json` cũ được
**sao chép** sang tên `_eval`, **không xoá, không ghi đè**.

In [ ]:
GRAD_DIR   = f'{SC_DIR}/metrics'
LEGACY_JS  = f'{GRAD_DIR}/initial_gradient_share.json'          # lan chay truoc (eval_tf)
EVAL_JS    = f'{GRAD_DIR}/initial_gradient_share_eval.json'
TRAIN_JS   = lambda s: f'{GRAD_DIR}/initial_gradient_share_train_s{s}.json'
AUG_SEEDS  = (42, 1337)                                         # KHOA: 2 seed tang cuong
AUG_WORKERS = 4
SWEEP_BS   = 32

def asl_neg_elem(z):
    """So hang AM cua ASL (y=0) — trung cong thuc ASLoss/WeightedASL cua nb12."""
    xp = torch.sigmoid(z); xn = (1 - xp + ASL_CLIP).clamp(max=1)
    return -torch.log(xn.clamp(min=ASL_EPS)) * torch.pow(1 - xn, ASL_GN)

def asl_neg_grad(z):
    # enable_grad() BAT BUOC: C9 goi ham nay ben trong `with torch.no_grad()`.
    # Neu thieu, asl_neg_elem(z) khong dung graph du z.requires_grad=True, va
    # torch.autograd.grad bao "element 0 ... does not have a grad_fn".
    with torch.enable_grad():
        z = z.detach().float().requires_grad_(True)
        return torch.autograd.grad(asl_neg_elem(z).sum(), z)[0].abs().detach()

# ---------------- doi chieu autograd vs dang dong ----------------
_zt = torch.linspace(-8, 8, 4001, device=dev)
_ga = asl_neg_grad(_zt)
_p = torch.sigmoid(_zt); _xn = (1 - _p + ASL_CLIP).clamp(max=1); _u = (_p - ASL_CLIP).clamp(min=0)
_cf = torch.where(_p > ASL_CLIP,
                  (_u**ASL_GN/_xn - ASL_GN*torch.log(_xn)*_u**(ASL_GN-1)) * _p*(1-_p),
                  torch.zeros_like(_p))
print(f'autograd vs dang dong |g|: max|diff| {(_ga-_cf).abs().max().item():.3e}  '
      f'{"OK" if (_ga-_cf).abs().max().item() < 1e-5 else "FAIL"}')
assert (_ga - _cf).abs().max().item() < 1e-5
with torch.no_grad(): _gi = asl_neg_grad(_zt)
print(f'asl_neg_grad chay duoc TRONG torch.no_grad() va cho ket qua y het: {torch.equal(_ga, _gi)}'
      f'  (nho enable_grad())')
assert torch.equal(_ga, _gi)
_r = (asl_neg_grad(torch.tensor([math.log(.9/.1)], device=dev)).item()
      / max(asl_neg_grad(torch.tensor([math.log(.1/.9)], device=dev)).item(), 1e-30))
print(f'g(p=0.05)={asl_neg_grad(torch.tensor([math.log(.05/.95)], device=dev)).item():.1e} (=0, bi clip)'
      f' | g(0.90)/g(0.10) = {_r:,.0f}x (~{math.log10(_r):.1f} bac)')
print('  -> (p-0.05)^4 CUC LOI. Khop theo SO O (V3.1) khong khop gradient — phai do g.')

# ---------------- loader tang cuong TAI LAP DUOC ----------------
def det_loader(tf, seed, bs, workers, split='train'):
    g = torch.Generator(); g.manual_seed(int(seed))
    def _wi(wid):
        s = (int(seed)*100_003 + wid) % (2**31 - 1)
        torch.manual_seed(s); np.random.seed(s); random.seed(s)
    torch.manual_seed(int(seed))                       # cho truong hop workers=0
    return DataLoader(ImgDS(IDX[split], tf), batch_size=bs, shuffle=False, collate_fn=collate,
                      num_workers=workers, pin_memory=(dev == 'cuda'), generator=g,
                      worker_init_fn=(_wi if workers > 0 else None), persistent_workers=False)

def _peek(seed, workers, k=3, bs=8):
    out = []
    for i, b in enumerate(det_loader(train_tf, seed, bs, workers)):
        out.append(b['image'].clone())
        if i + 1 >= k: break
    return out

print(f'\n=== KIEM TANG CUONG TAI LAP DUOC (workers={AUG_WORKERS}) ===')
for _w in (AUG_WORKERS, 0):
    _a1, _a2, _b1 = _peek(AUG_SEEDS[0], _w), _peek(AUG_SEEDS[0], _w), _peek(AUG_SEEDS[1], _w)
    _det = all(torch.equal(x, y) for x, y in zip(_a1, _a2))
    _dif = any(not torch.equal(x, y) for x, y in zip(_a1, _b1))
    print(f'  workers={_w}: cung seed -> bit-for-bit {_det} | khac seed -> anh khac {_dif}')
    if _det and _dif:
        AUG_WORKERS = _w; break
else:
    raise RuntimeError('khong tao duoc tang cuong tai lap duoc -> khong the khoa beta')
print(f'  -> dung workers={AUG_WORKERS}')

# ---------------- mot luot quet ----------------
def sweep_gradient(tf, seed, workers, tag, bs=SWEEP_BS):
    _A.eval()
    _ct = torch.from_numpy(CONTESTED.astype('float32')).to(dev)
    _ot = torch.from_numpy((IS_BLANK & ~CONTESTED).astype('float32')).to(dev)
    Gc = torch.zeros(C, dtype=torch.float64, device=dev)
    Go = torch.zeros(C, dtype=torch.float64, device=dev)
    acc, t0 = {l: {} for l in RARE3}, time.time()
    for b in tqdm(det_loader(tf, seed, bs, workers), desc=f'0b {tag}'):
        b = to_dev(b)
        with torch.no_grad(): z = _A(forward_shared(b['image']))
        g = asl_neg_grad(z); sr = b['sr']
        gc = g*_ct[sr]; Gc += gc.sum(0).double(); Go += (g*_ot[sr]).sum(0).double()
        _s = SUBJ[sr.cpu().numpy()]; _v = gc[:, RARE3_IDX].double().cpu().numpy()   # [B, 3]
        for _r, _t in zip(*np.nonzero(_v > 0)):
            _d = acc[RARE3[_t]]; _k = int(_s[_r])
            _d[_k] = _d.get(_k, 0.0) + float(_v[_r, _t])
    return dict(tag=tag, seed=int(seed), workers=int(workers), batch=int(bs),
                transform=('train_tf' if tf is train_tf else 'eval_tf'),
                seconds=time.time()-t0, n_images=int(len(IDX['train'])),
                G_cont=Gc.cpu().numpy().tolist(), G_ord=Go.cpu().numpy().tolist(),
                subj_cont_mass={l: {str(k): v for k, v in d.items()} for l, d in acc.items()})

def cached(path, fn):
    if os.path.exists(path):
        r = json.load(open(path)); print(f'  <- cache {os.path.basename(path)}'); return r
    r = fn(); json.dump(r, open(path, 'w'))
    print(f'  -> {os.path.basename(path)}  ({r["seconds"]/60:.1f} phut)'); return r

print('\n=== QUET GRADIENT ===')
if not os.path.exists(EVAL_JS) and os.path.exists(LEGACY_JS):
    shutil.copyfile(LEGACY_JS, EVAL_JS)     # SAO CHEP, khong xoa/ghi de file cu
    print(f'  migrate {os.path.basename(LEGACY_JS)} -> {os.path.basename(EVAL_JS)} (giu file cu)')
R_EVAL = cached(EVAL_JS, lambda: sweep_gradient(eval_tf, 0, 0, 'eval_tf'))
R_TR   = [cached(TRAIN_JS(s), (lambda s=s: sweep_gradient(train_tf, s, AUG_WORKERS, f'train_tf s{s}')))
          for s in AUG_SEEDS]

beta_of = lambda Gc, Go: np.where(np.asarray(Gc)+np.asarray(Go) > 0,
    (ETA*np.asarray(Gc)+np.asarray(Go))/np.maximum(np.asarray(Gc)+np.asarray(Go), 1e-30), 1.0)
GC_TR = np.sum([r['G_cont'] for r in R_TR], 0); GO_TR = np.sum([r['G_ord'] for r in R_TR], 0)
GC_EV = np.asarray(R_EVAL['G_cont']);           GO_EV = np.asarray(R_EVAL['G_ord'])
BETA_TRAIN, BETA_EVAL = beta_of(GC_TR, GO_TR), beta_of(GC_EV, GO_EV)
BETA = BETA_TRAIN                                        # <<<< PRIMARY
B_PER = [beta_of(r['G_cont'], r['G_ord']) for r in R_TR]

# ---------------- bang chinh ----------------
_nb_tr = (IS_BLANK & IS_TRAIN[:, None]).sum(0)           # mau so TRAIN-ONLY (sua loi ban dau)
GS = pd.DataFrame(dict(label=LABELS, tier=TIER, rare3=[l in RARE3 for l in LABELS],
    n_contested=[int(CONTESTED[:, j].sum()) for j in range(C)], n_blank_train=_nb_tr,
    G_cont=GC_TR, G_ord=GO_TR, cont_share=GC_TR/np.maximum(GC_TR+GO_TR, 1e-30),
    beta_train=BETA_TRAIN, beta_eval=BETA_EVAL, d_beta=BETA_TRAIN-BETA_EVAL))
GS['cell_share'] = GS.n_contested/GS.n_blank_train.clip(lower=1)
GS['enrich'] = GS.cont_share/GS.cell_share.clip(lower=1e-12)
GS.to_csv(f'{GRAD_DIR}/gradient_calibration.csv', index=False)
print('\n=== GRADIENT SHARE tai INIT — PRIMARY = train_tf pooled 2 seed ===')
print(GS.to_string(index=False, float_format=lambda v: f'{v:.6f}'))
assert np.allclose(BETA[NONTGT_IDX], 1.0), 'NonTarget-10 phai co beta = 1'
assert ((BETA >= ETA-1e-9) & (BETA <= 1+1e-9)).all(), 'beta ngoai [eta, 1]'
print(f'\n  NonTarget-10 beta == 1 OK | beta thuoc [{BETA.min():.6f}, {BETA.max():.6f}] c [{ETA}, 1] OK')
print('  n_blank_train la mau so TRAIN-ONLY (ban dau toi dung mau so toan cohort -> enrich bi thoi ~1.5x)')

# ---------------- train vs eval, va do nhay theo seed tang cuong ----------------
_r3 = GS[GS.rare3]
print('\n=== beta_train vs beta_eval, va NHIEU giua 2 augmentation seed ===')
_cmp = pd.DataFrame(dict(label=[LABELS[j] for j in RARE3_IDX],
    beta_s42=[B_PER[0][j] for j in RARE3_IDX], beta_s1337=[B_PER[1][j] for j in RARE3_IDX],
    beta_train=[BETA_TRAIN[j] for j in RARE3_IDX], beta_eval=[BETA_EVAL[j] for j in RARE3_IDX]))
_cmp['seed_spread'] = (_cmp.beta_s42-_cmp.beta_s1337).abs()
_cmp['train_vs_eval'] = (_cmp.beta_train-_cmp.beta_eval).abs()
print(_cmp.to_string(index=False, float_format=lambda v: f'{v:.6f}'))
_ss, _te = float(_cmp.seed_spread.max()), float(_cmp.train_vs_eval.max())
print(f'\n  max |beta(s42) - beta(s1337)|   = {_ss:.6f}   <- nhieu noi tai cua tang cuong')
print(f'  max |beta_train - beta_eval|    = {_te:.6f}   <- hieu ung he thong cua tang cuong')
print(f'''  {"train_vs_eval > seed_spread -> tang cuong CO tac dong he thong, dung beta_train la dung."
        if _te > _ss else
        "train_vs_eval <= seed_spread -> hieu ung eval-vs-train KHONG tach duoc khoi nhieu tang cuong;"}
  {"" if _te > _ss else "  beta_train van la primary (dung phan phoi training that), nhung tranh luan nay la moot."}''')

print('\n=== cell share vs GRADIENT share (cho V3.1 sai) — mau so TRAIN-ONLY ===')
print(_r3[['label','n_contested','n_blank_train','cell_share','cont_share','enrich']]
      .to_string(index=False, float_format=lambda v: f'{v:.6f}'))
print(f'  enrich != 1.00 -> khop theo so o KHONG khop gradient.')

# ---------------- co can thiep ----------------
_tot_blank = float((GC_TR+GO_TR).sum()); _r3m = float((GC_TR+GO_TR)[RARE3_IDX].sum())
_removed = float((1-ETA)*GC_TR[RARE3_IDX].sum())
print(f'''
=== CO CAN THIEP (doc truoc khi cam ket ~40 gio GPU) ===
  tong negative-gradient mass cua o BLANK, 13 nhan : {_tot_blank:12,.0f}
  phan cua Rare-3                                   : {_r3m:12,.0f}  ({_r3m/_tot_blank:.2%})
  R2 xoa (1-eta)*G_cont cua Rare-3                  : {_removed:12,.0f}  ({_removed/_tot_blank:.2%} tong)
  -> toan bo thi nghiem dich {_removed/_tot_blank:.2%} khoi luong gradient am.
     Dat canh REVIEW_LABEL_NOISE_24H: nhieu nhan dich Rare-3 mAP +/-0.05, cong dang xet +0.002.
     Day KHONG phai ly do dung; day la co hieu ung ky vong, va no lam kich ban
     "D1 ~ 0 & D2 ~ 0" thanh ket cuc xac suat cao nhat — dung nhu Q4 da du lieu truoc.''')

# ---------------- concentration gate ----------------
def norm_subj(r):
    """-> {label: {subject: mass}}. File legacy (eval) chi co dang gop -> '_pooled'."""
    m = r['subj_cont_mass']
    return m if (m and isinstance(next(iter(m.values())), dict)) else {'_pooled': m}

def subj_series(rs, label=None):
    d = {}
    for r in rs:
        m = norm_subj(r)
        srcs = ([m[label]] if label in m else []) if label is not None else list(m.values())
        for src in srcs:
            for k, v in src.items(): d[k] = d.get(k, 0.0) + v
    return pd.Series(d, dtype='float64').sort_values(ascending=False)

def top1(sc):
    if not len(sc): return 0.0, 0, 0
    k = max(1, int(np.ceil(0.01*len(sc))))
    return float(sc.iloc[:k].sum()/max(float(sc.sum()), 1e-30)), k, len(sc)

SUBJ_CONT = subj_series(R_TR)                                  # gop Rare-3, train_tf = PRIMARY
_t_tr, _k_tr, _n_tr = top1(SUBJ_CONT)
_t_ev, _k_ev, _n_ev = top1(subj_series([R_EVAL]))
print()
print(f'=== CONCENTRATION GATE — theo GRADIENT MASS (§4.4) ===')
print(f'  PRIMARY  train_tf gop Rare-3 : {_n_tr:,} subject | top 1% = {_k_tr} -> {_t_tr:.4%} '
      f'(nguong <= {GATE_TOP1PCT_SHARE:.0%})  {"OK" if _t_tr <= GATE_TOP1PCT_SHARE else "FAIL"}')
print(f'  sensitivity eval_tf          : {_n_ev:,} subject | top 1% = {_k_ev} -> {_t_ev:.4%}  '
      f'{"OK" if _t_ev <= GATE_TOP1PCT_SHARE else "OK/FAIL (sensitivity)"}')
_PC = []
for l in RARE3:
    sc = subj_series(R_TR, l); t_, k_, n_ = top1(sc)
    _PC.append(dict(label=l, subjects=n_, top1pct_k=k_, top1pct_share=t_,
                    over_threshold=bool(t_ > GATE_TOP1PCT_SHARE)))
PC = pd.DataFrame(_PC); PC.to_csv(f'{GRAD_DIR}/concentration_by_class.csv', index=False)
print()
print('  --- DIAGNOSTIC: concentration TUNG NHAN (gate khoa truoc la so GOP, khong phai so nay) ---')
print(PC.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
if PC.over_threshold.any():
    print(f'  ! {list(PC[PC.over_threshold].label)} vuot {GATE_TOP1PCT_SHARE:.0%} khi xet RIENG.')
    print('    Gate khoa truoc (§4.4) la mot nguong duy nhat -> doc tu nhien la so GOP, va so gop PASS.')
    print('    KHONG tu dong fail o day: dat them tieu chi sau khi thay so la doi gate hau nghiem.')
    print('    Nhung phai GHI vao bao cao: nhan do co bang chung tap trung o it benh nhan hon.')
else:
    print('  khong nhan nao vuot nguong khi xet rieng.')
_part = json.load(open(f'{SC_DIR}/metrics/sufficiency_partial.json'))
_ok = _part['gate_cells_subjects_pass'] and _t_tr <= GATE_TOP1PCT_SHARE
json.dump(dict(**_part, primary='train_tf pooled seeds ' + str(list(AUG_SEEDS)),
               top1pct_gradient_share_train=_t_tr, top1pct_gradient_share_eval=_t_ev,
               gate_pass=bool(_ok), eta=ETA, aug_workers=int(AUG_WORKERS),
               beta_train={l: float(b) for l, b in zip(LABELS, BETA_TRAIN)},
               beta_eval={l: float(b) for l, b in zip(LABELS, BETA_EVAL)},
               beta_per_aug_seed={str(s): {l: float(b) for l, b in zip(LABELS, bp)}
                                  for s, bp in zip(AUG_SEEDS, B_PER)},
               max_seed_spread=_ss, max_train_vs_eval=_te,
               intervention_share_of_blank_negative_mass=_removed/_tot_blank),
          open(f'{GRAD_DIR}/gradient_calibration.json', 'w'), indent=1)
print(f'''
=== GO / NO-GO cua 0a + 0b: {"GO" if _ok else "NO-GO"} ===
  {"Du dieu kien de R2 la primary. Sang 0c (C9)." if _ok else
   "R2 MAT primary status (§4.4). KHONG duoc sua manifest roi tiep tuc claim cung estimand."}
  beta PRIMARY = beta_train (pooled 2 aug seed). beta_eval giu lam sensitivity.''')

## C9 · Weight matrix + probe throughput + smoke 200 step (**0c** + **0d**)

`W[arm]` là `[N_study, 13]`, dùng **`β_train`**; mỗi ảnh thừa hưởng weight của study nó thuộc về.

```
R0     : 1 khap noi
R1GM   : beta_train[c] cho MOI o blank (contested + ordinary), 1 cho o khac
R2     : eta cho o CONTESTED blank, 1 cho o khac
```

Chuẩn hoá bằng **hằng số** `B_actual × C`. Với `w ≡ 1` đây **đúng bằng** `ASLoss(...).mean()` của
nb11–15 ⇒ `lr = 1e-5` vẫn ở đúng thang cũ. Không chia `Σw`.

### Probe batch theo **throughput**, không chỉ VRAM *(chốt 2026-07-30)*

Lần trước probe dừng ở `B = 32` chỉ vì hết danh sách — peak mới `14.4/102 GB (14%)`. Giờ probe
`32/64/128/256`, **đo s/step thật** ở mỗi mức (2 warm-up + 5 lần đo, có `cuda.synchronize`), rồi
chọn `B` cho **img/s cao nhất** trong số các mức còn dưới 80% VRAM.

`EFF = 256` giữ nguyên nên `accum = 256/B`: số optimizer step **không đổi** theo `B`. Batch lớn
chỉ đổi hiệu suất kernel, không đổi tổng FLOP — nên đừng kỳ vọng gấp bội.

Probe cho **throughput GPU thuần** (lặp lại một batch, không qua dataloader). Con số thật là
`s/step` của smoke 200 step chạy loader thật.

### Khớp gradient: **pooled + bootstrap CI** *(chốt 2026-07-30)*

Bản đầu tôi lấy `step 0` (một batch 32 ảnh, ~5 ô contested) làm số được claim. Sai: `g` trải
hơn 5 bậc nên tổng của ~5 số hạng đuôi nặng gần như thuần nhiễu.

Giờ: **pooled trên toàn bộ 200 batch + bootstrap CI** (`N_BOOT`, resample theo batch). Single
batch chỉ còn là sanity check. Ghi cả hai phiên bản:

| | gồm | vì sao |
|---|---|---|
| `all-negative` | mọi ô `y=0` của Rare-3 | gồm cả ô `raw==0` (`w=1` ở mọi arm) ⇒ pha loãng, kém nhạy |
| **`blank-only`** | chỉ ô blank | **đúng tập mà `β` khớp** ⇒ đây là phép thử sắc |

Tiêu chí đọc: CI của tỉ số **chứa 1.0**.

### Hard failure — danh sách **dương** 6 mục

| kiểm | tiêu chí |
|---|---|
| blocks 0–8 | **bất biến bit-for-bit** |
| blocks 9–11 + `backbone.layernorm` | **có** update |
| H5 `norm`/`mld` ở ep1 | **bất biến** (head warmup) |
| loss, grad norm | hữu hạn |
| output | không thành hằng số |
| optimizer/scaler | không hỏng |

Drift / cosine / rank là **DIAGNOSTIC**, không dùng để dừng run hay loại checkpoint (§7).

### Runtime: cộng cả **val pass**

Ước lượng cũ `~35 h` **chưa gồm** predict trên VAL mỗi epoch. Giờ đo luôn một lượt val 3 arm để
con số cam kết là số đo, không phải số đoán.

In [ ]:
# ---------------- weight matrix (dung BETA = beta_train) ----------------
def build_W(arm):
    W = np.ones((N, C), 'float32')
    if arm == 'R0':   return W
    if arm == 'R1GM': W[IS_BLANK] = np.broadcast_to(BETA.astype('float32'), (N, C))[IS_BLANK]
    if arm == 'R2':   W[CONTESTED & IS_BLANK] = ETA
    return W
W  = {a: build_W(a) for a in ARMS}
Wt = {a: torch.from_numpy(W[a]).to(dev) for a in ARMS}
BLANK_T = torch.from_numpy(IS_BLANK.astype('float32')).to(dev)
print('=== weight matrix (beta = beta_train) ===')
for a in ARMS:
    w = W[a]
    print(f'  {a:<5} uniq {np.unique(np.round(w,6))[:6]}... | mean {w.mean():.6f} | '
          f'n(w!=1) {int((w!=1).sum()):>9,} | n(w=={ETA}) {int(np.isclose(w,ETA).sum()):>8,}')
assert (W['R0'] == 1).all()
assert (W['R2'][~(CONTESTED & IS_BLANK)] == 1).all() and np.isclose(W['R2'][CONTESTED & IS_BLANK], ETA).all()
assert (W['R1GM'][~IS_BLANK] == 1).all()
assert np.allclose(W['R1GM'][:, NONTGT_IDX], 1.0), 'NonTarget-10 bi doi weight o R1GM -> control sai'
print(f'  NonTarget-10 weight == 1 o CA BA arm: '
      f'{all(np.allclose(W[a][:, NONTGT_IDX], 1.0) for a in ARMS)}')
print('  -> can thiep chi cham 3 nhan Rare-3; 10 nhan con lai la placebo channel.')

class CellWASL(nn.Module):
    """ASL + weight tung O. w==1 => trung ASLoss().mean() cua nb11-15 (chuan hoa HANG SO B*C)."""
    def __init__(s): super().__init__(); s.gn, s.gp, s.clip, s.eps = ASL_GN, ASL_GP, ASL_CLIP, ASL_EPS
    def forward(s, x, y, w, parts=False):
        y = y.float(); xp = torch.sigmoid(x); xn = (1-xp+s.clip).clamp(max=1)
        lp = y*torch.log(xp.clamp(min=s.eps)); ln = (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y); mod = torch.pow(1-pt, g)
        Lp, Ln = -lp*mod, -ln*mod
        loss = (w*(Lp+Ln)).sum()/(x.shape[0]*x.shape[1])
        if parts: return loss, (w*Lp).mean(0).detach(), (w*Ln).mean(0).detach()
        return loss
CRIT = CellWASL().to(dev)

class _ASLref(nn.Module):
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = (1-xp+ASL_CLIP).clamp(max=1)
        los = y*torch.log(xp.clamp(min=ASL_EPS)) + (1-y)*torch.log(xn.clamp(min=ASL_EPS))
        pt = xp*y + xn*(1-y); g = ASL_GP*y + ASL_GN*(1-y)
        return -(los*torch.pow(1-pt, g)).mean()
torch.manual_seed(0)
_x = torch.randn(64, C, device=dev); _y = (torch.rand(64, C, device=dev) < .15).float()
_rf = _ASLref()(_x, _y); _nw = CRIT(_x, _y, torch.ones_like(_x))
print(f'\nCellWASL(w=1) == ASLoss cua nb11-15: {_rf.item():.10f} vs {_nw.item():.10f} '
      f'| diff {abs(_rf.item()-_nw.item()):.3e} {"OK" if torch.equal(_rf,_nw) else "FAIL"}')
assert torch.equal(_rf, _nw), 'w=1 khong tai tao ASL -> hang so chuan hoa da doi, lr sai thang'

# ---------------- optimizer / sched / clip ----------------
def make_opt(m):
    def sp(ps, lr):
        d = [p for p in ps if p.ndim > 1]; nd = [p for p in ps if p.ndim <= 1]
        return [{'params': d, 'lr': lr, 'weight_decay': WD},
                {'params': nd, 'lr': lr, 'weight_decay': 0.0}]
    return torch.optim.AdamW(sp(tail_params(m), LR_BLOCKS) + sp(head_params(m), LR_HEAD))

def make_sched(opt, total, warm):
    cw = lambda s: s/max(warm,1) if s < warm else 0.5*(1+math.cos(math.pi*(s-warm)/max(total-warm,1)))
    return torch.optim.lr_scheduler.LambdaLR(opt, [cw]*len(opt.param_groups))

set_head_trainable = lambda m, on: [p.requires_grad_(on) for p in head_params(m)]

def clip_arm(m):
    t = [p for p in tail_params(m) if p.grad is not None]
    h = [p for p in head_params(m) if p.grad is not None]
    return (float(nn.utils.clip_grad_norm_(t, CLIP_BLOCKS)) if t else 0.0,
            float(nn.utils.clip_grad_norm_(h, CLIP_HEAD)) if h else 0.0)

def sync_rng(ep, i):
    s = (SEED*1_000_003 + ep*100_003 + i) % (2**31-1)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# ---------------- probe: THROUGHPUT + VRAM ----------------
def probe_batch(cands=(32, 64, 128, 256), warm=2, meas=5):
    if dev != 'cuda': return 2, pd.DataFrame()
    A = [build_arm() for _ in ARMS]
    for m in A: set_head_trainable(m, False); m.train()      # dung policy ep1
    tot = torch.cuda.get_device_properties(0).total_memory/1e9
    rows = []
    for B in cands:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            bb = to_dev(collate([ImgDS(IDX['train'], train_tf)[k] for k in range(B)]))
            for it in range(warm+meas):
                if it == warm: torch.cuda.synchronize(); t0 = time.time()
                z = forward_shared(bb['image'])
                for m in A:
                    CRIT(m(z), bb['y'], Wt['R0'][bb['sr']]).backward(); m.zero_grad(set_to_none=True)
            torch.cuda.synchronize()
            dt = (time.time()-t0)/meas; pk = torch.cuda.max_memory_allocated()/1e9
            rows.append(dict(batch=B, s_per_step=dt, img_per_s=B/dt, peak_gb=pk,
                             frac=pk/tot, ok=bool(pk/tot < 0.80)))
            print(f'  B={B:4d}  {dt:6.3f} s/step  {B/dt:7.1f} img/s  peak {pk:5.1f}/{tot:.0f} GB '
                  f'({pk/tot:5.1%})  {"OK" if pk/tot < 0.80 else "QUA 80% -> dung"}')
            del bb
            if pk/tot >= 0.80: break
        except torch.cuda.OutOfMemoryError:
            print(f'  B={B:4d}  OOM'); torch.cuda.empty_cache(); break
    del A; torch.cuda.empty_cache()
    PB = pd.DataFrame(rows); _ok = PB[PB.ok]
    return (int(_ok.loc[_ok.img_per_s.idxmax(), 'batch']) if len(_ok) else cands[0]), PB

PB_CSV = f'{SC_DIR}/metrics/probe_batch.csv'
print()
print(f'=== PROBE BATCH ({len(ARMS)} arm x 3 block trainable x {N_PATCH+1} token) ===')
if os.path.exists(PB_CSV):
    PB = pd.read_csv(PB_CSV); _o = PB[PB.ok]
    BATCH = int(_o.loc[_o.img_per_s.idxmax(), 'batch']) if len(_o) else 32
    print(f'  <- cache {os.path.basename(PB_CSV)} (khong chay lai probe)')
    print(PB.to_string(index=False, float_format=lambda v: f'{v:.4g}'))
else:
    BATCH, PB = probe_batch(); PB.to_csv(PB_CSV, index=False)
ACCUM = max(1, EFF//BATCH)
if len(PB):
    _best = PB.loc[PB.batch == BATCH].iloc[0]
    print(f'-> chon BATCH {BATCH} theo throughput ({_best.img_per_s:.1f} img/s, '
          f'{_best.frac:.1%} VRAM) | accum {ACCUM} -> effective {BATCH*ACCUM} (muc tieu {EFF})')
    print(f'   nhanh hon B=32: {_best.img_per_s/PB.iloc[0].img_per_s:.2f}x')
assert BATCH*ACCUM == EFF or BATCH > EFF, f'BATCH*ACCUM={BATCH*ACCUM} != EFF={EFF}'

# ---------------- smoke 200 step ----------------
SMOKE_STEPS = 200
def smoke(steps=SMOKE_STEPS, batch=BATCH):
    base = build_arm(SEED); ST = {}
    for a in ARMS:
        m = Arm().to(dev); m.load_state_dict({k: v.detach().clone() for k, v in base.state_dict().items()})
        o = make_opt(m)
        ST[a] = dict(m=m, opt=o, sch=make_sched(o, steps//ACCUM+1, max(1, int(WARMUP_FRAC*steps//ACCUM))))
    _sd = [ST[a]['m'].state_dict() for a in ARMS]
    assert all(torch.equal(x, y) for d in _sd[1:] for x, y in zip(_sd[0].values(), d.values())), \
        'cac arm KHONG cung init -> khong con la paired'
    print(f'{len(ARMS)} arm cung init bit-for-bit')
    FROZEN0 = {k: v.detach().clone() for k, v in backbone.state_dict().items()}
    INIT = {a: {k: v.detach().clone() for k, v in ST[a]['m'].state_dict().items()} for a in ARMS}
    for a in ARMS: set_head_trainable(ST[a]['m'], False); ST[a]['m'].train()   # ep1: head frozen
    print(f'ep1 policy: head frozen | tail trainable ({_np(tail_params(ST[ARMS[0]]["m"])):,} params)')

    dl = loader('train', train_tf, True, batch, workers=4)
    LOGS = {a: [] for a in ARMS}; GM = []
    torch.manual_seed(SEED*7919 + 0)
    it = iter(dl); t0 = time.time()
    if dev == 'cuda': torch.cuda.reset_peak_memory_stats()
    for i in tqdm(range(steps), desc='0c smoke'):
        try: b = next(it)
        except StopIteration: it = iter(dl); b = next(it)
        b = to_dev(b); z = forward_shared(b['image'])
        step_now = ((i+1) % ACCUM == 0); gm = {}
        for a in ARMS:
            e = ST[a]
            if STOCHASTIC: sync_rng(0, i)
            zz = e['m'](z)
            loss = CRIT(zz, b['y'], Wt[a][b['sr']])
            assert torch.isfinite(loss), f'HARD FAILURE: loss khong huu han o arm {a} step {i}'
            (loss/ACCUM).backward()
            _g = asl_neg_grad(zz)*(1-b['y'])*Wt[a][b['sr']]
            gm[a] = float(_g[:, RARE3_IDX].sum())
            gm[a+'_blank'] = float((_g*BLANK_T[b['sr']])[:, RARE3_IDX].sum())
            if step_now:
                nt, nh = clip_arm(e['m'])
                assert math.isfinite(nt) and math.isfinite(nh), f'HARD FAILURE: grad norm {a} step {i}'
                e['opt'].step(); e['sch'].step(); e['opt'].zero_grad(set_to_none=True)
                LOGS[a].append(dict(step=i+1, loss=float(loss.detach()), gn_tail=nt, gn_head=nh,
                                    lr=e['opt'].param_groups[0]['lr']))
        GM.append(gm)
    el = time.time() - t0

    # ---- HARD FAILURE (danh sach duong 6 muc) ----
    _fn = lambda ps: float(torch.linalg.vector_norm(
        torch.stack([torch.linalg.vector_norm(p.detach().float()) for p in ps])))
    print(f'\n=== HARD FAILURE CHECKS (danh sach duong, §7 · REVIEW §2·Q3) ===')
    _bad = {k: (FROZEN0[k]-v).abs().max().item() for k, v in backbone.state_dict().items()
            if not torch.equal(FROZEN0[k], v)}
    print(f'  1. blocks 0-8 bat bien bit-for-bit         {"OK" if not _bad else f"FAIL {list(_bad)[:3]}"}')
    assert not _bad, f'HARD FAILURE: shared frozen backbone bi doi: {list(_bad)[:5]}'
    R = []
    for a in ARMS:
        sd = ST[a]['m'].state_dict()
        dt = max((sd[k]-INIT[a][k]).abs().max().item() for k in sd if k.startswith('tail.'))
        dh = max((sd[k]-INIT[a][k]).abs().max().item() for k in sd if k.startswith('head.'))
        dln = max((sd[k]-INIT[a][k]).abs().max().item() for k in sd if k.startswith('tail.ln.'))
        rel = float(torch.linalg.vector_norm(torch.stack([(sd[k]-INIT[a][k]).float().flatten().norm()
                    for k in sd if k.startswith('tail.')]))) / max(_fn([INIT[a][k] for k in sd
                    if k.startswith('tail.')]), 1e-12)
        cs = [float(F.cosine_similarity(sd[k].float().flatten(), INIT[a][k].float().flatten(), 0))
              for k in sd if k.startswith('tail.') and sd[k].ndim > 1]
        R.append(dict(arm=a, d_tail=dt, d_tail_ln=dln, d_head=dh, rel_drift_tail=rel,
                      min_cos_tail=float(np.min(cs)), final_loss=LOGS[a][-1]['loss'],
                      gn_tail=float(np.mean([r['gn_tail'] for r in LOGS[a][-20:]])),
                      gn_head=float(np.mean([r['gn_head'] for r in LOGS[a][-20:]]))))
    RD = pd.DataFrame(R)
    for n_, c_, ok_ in ((2, 'blocks 9-11 CO update (d_tail > 0)', (RD.d_tail > 0).all()),
                        (3, 'backbone.layernorm CO update', (RD.d_tail_ln > 0).all()),
                        (4, 'H5 head bat bien o ep1 (d_head == 0)', (RD.d_head == 0).all()),
                        (5, 'loss & grad norm huu han', True),
                        (6, 'output khong thanh hang so', (RD.min_cos_tail > 0.9).all())):
        print(f'  {n_}. {c_:<41} {"OK" if ok_ else "FAIL"}')
    assert (RD.d_tail > 0).all() and (RD.d_tail_ln > 0).all(), 'HARD FAILURE: tail khong update'
    assert (RD.d_head == 0).all(), 'HARD FAILURE: head bi update trong head-warmup epoch'
    print(f'\n=== DIAGNOSTIC (KHONG dung de dung run / loai checkpoint — §7) ===')
    print(RD.to_string(index=False, float_format=lambda v: f'{v:.6g}'))

    # ---- 0d: khop gradient, POOLED + BOOTSTRAP CI ----
    _rng = np.random.default_rng(SEED); _ix = _rng.integers(0, len(GM), (N_BOOT, len(GM)))
    def _pooled(k1, k2):
        a = np.array([g[k1] for g in GM]); b = np.array([g[k2] for g in GM])
        pt = float(a.sum()/max(b.sum(), 1e-30))
        bo = a[_ix].sum(1)/np.maximum(b[_ix].sum(1), 1e-30)
        return pt, float(np.percentile(bo, 2.5)), float(np.percentile(bo, 97.5))
    print(f'\n=== 0d · KHOP GRADIENT — POOLED {len(GM)} batch + bootstrap CI ({N_BOOT}) ===')
    CI = {}
    for nm, k1, k2 in (('blank-only  (phep thu SAC — dung tap ma beta khop)', 'R1GM_blank', 'R2_blank'),
                       ('all-negative (gom o raw==0, w=1 moi arm -> pha loang)', 'R1GM', 'R2')):
        pt, lo, hi = _pooled(k1, k2); CI[k1] = dict(ratio=pt, lo=lo, hi=hi)
        print(f'  R1GM/R2 {nm:<52} {pt:.6f}  CI95 [{lo:.6f}, {hi:.6f}]  '
              f'{"CHUA 1.0 -> khop" if lo <= 1.0 <= hi else "KHONG chua 1.0"}')
    _s0 = GM[0]['R1GM_blank']/max(GM[0]['R2_blank'], 1e-30)
    print(f'  sanity mot batch (step 0): {_s0:.6f}  <- CHI la sanity check.')
    print(f'    ~{batch} anh/batch => rat it o contested, ma g trai >5 bac => phuong sai lon.')
    print(f'    So duoc CLAIM la pooled + CI o tren, KHONG phai so nay.')
    _m = np.array([[g['R0'], g['R1GM'], g['R2']] for g in GM]).mean(0)
    print(f'  mass trung binh (all-neg): R0 {_m[0]:.4f} | R1GM {_m[1]:.4f} | R2 {_m[2]:.4f}'
          f'  | R1GM/R0 {_m[1]/_m[0]:.4f} | R2/R0 {_m[2]/_m[0]:.4f}')

    # ---- runtime: train + VAL ----
    _spe = math.ceil(len(IDX['train'])/batch)
    h_tr = _spe*el/steps/3600
    for a in ARMS: ST[a]['m'].eval()
    torch.cuda.synchronize() if dev == 'cuda' else None
    tv = time.time(); nv = 0
    with torch.no_grad():
        for b in tqdm(loader('val', eval_tf, False, batch, workers=4), desc='val pass', leave=False):
            b = to_dev(b); z = forward_shared(b['image'])
            for a in ARMS: torch.sigmoid(ST[a]['m'](z))
            nv += b['image'].shape[0]
    torch.cuda.synchronize() if dev == 'cuda' else None
    h_val = (time.time()-tv)/3600
    print(f'\n=== RUNTIME (do that, gom ca VAL) ===')
    print(f'  smoke {steps} step x {len(ARMS)} arm @B={batch} | {el:.0f}s | {el/steps:.3f} s/step '
          f'| {batch/(el/steps):.1f} img/s')
    if dev == 'cuda':
        print(f'  peak VRAM {torch.cuda.max_memory_allocated()/1e9:.1f} / '
              f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
    print(f'  val pass  {nv:,} anh x {len(ARMS)} arm | {h_val*3600:.0f}s')
    print(f'  1 epoch = train {h_tr:.2f} h + val {h_val:.2f} h = {h_tr+h_val:.2f} h '
          f'({_spe:,} step/epoch)')
    print(f'  {EPOCHS} epoch -> {EPOCHS*(h_tr+h_val):.1f} h/seed  ->  3 seed = '
          f'{3*EPOCHS*(h_tr+h_val):.0f} h  <<< CON SO CAM KET')
    _snap = _np(tail_params(ST[ARMS[0]]['m']) + head_params(ST[ARMS[0]]['m']))*4/1e9
    print(f'  snapshot {_snap*len(ARMS):.2f} GB/epoch ({len(ARMS)} arm) -> '
          f'{_snap*len(ARMS)*EPOCHS:.1f} GB/seed -> {_snap*len(ARMS)*EPOCHS*3:.0f} GB (3 seed)')
    print(f'  resume (params + 2 moment AdamW) {_snap*len(ARMS)*3:.2f} GB, ghi de moi epoch')
    print(f'  -> {EPOCHS*(h_tr+h_val):.1f} h/seed VUOT XA mot session Colab: '
          f'resume theo epoch la BAT BUOC trong train loop.')
    json.dump(dict(steps=steps, batch=batch, accum=ACCUM, seconds=el, s_per_step=el/steps,
                   img_per_s=batch/(el/steps), steps_per_epoch=_spe,
                   hours_train_per_epoch=h_tr, hours_val_per_epoch=h_val,
                   hours_per_seed=EPOCHS*(h_tr+h_val), hours_3seed=3*EPOCHS*(h_tr+h_val),
                   n_val_images=int(nv), snapshot_gb_per_epoch=_snap*len(ARMS),
                   peak_vram_gb=(torch.cuda.max_memory_allocated()/1e9 if dev == 'cuda' else None),
                   grad_match=CI, sanity_single_batch=_s0, n_boot=N_BOOT,
                   beta_source='train_tf pooled aug seeds ' + str(list(AUG_SEEDS)),
                   gm_per_batch=GM, diagnostics=RD.to_dict('records'),
                   note='drift/cosine la DIAGNOSTIC ONLY (§7). Claim = pooled ratio + CI, khong phai step 0.'),
              open(f'{SC_DIR}/metrics/smoke_200step.json', 'w'), indent=1)
    del ST, base, INIT, FROZEN0
    torch.cuda.empty_cache() if dev == 'cuda' else None
    return RD

SMOKE_JS = f'{SC_DIR}/metrics/smoke_200step.json'
FORCE_SMOKE = False          # True de chay lai (SE GHI DE so da khoa o C10)
if os.path.exists(SMOKE_JS) and not FORCE_SMOKE:
    _s = json.load(open(SMOKE_JS)); SMOKE = pd.DataFrame(_s['diagnostics'])
    print(f'smoke <- cache {os.path.basename(SMOKE_JS)} (KHONG chay lai)')
    print(f'  6/6 hard-failure da pass o lan chay truoc; day la ban ghi, khong phai do moi.')
    for k, v in _s['grad_match'].items():
        print(f'  R1GM/R2 {k:<12} {v["ratio"]:.6f}  CI95 [{v["lo"]:.6f}, {v["hi"]:.6f}]')
    print(f'  batch {_s["batch"]} | {_s["s_per_step"]:.3f} s/step | {_s["img_per_s"]:.1f} img/s '
          f'| peak {_s["peak_vram_gb"]:.1f} GB')
    print(f'  1 epoch {_s["hours_train_per_epoch"]+_s["hours_val_per_epoch"]:.2f} h -> '
          f'{_s["hours_per_seed"]:.1f} h/seed -> {_s["hours_3seed"]:.0f} h (3 seed)')
    print(SMOKE.to_string(index=False, float_format=lambda v: f'{v:.6g}'))
    print(f'  ! Dat FORCE_SMOKE=True de do lai — nhung no GHI DE smoke_200step.json,')
    print(f'    tuc doi so da duoc C10 khoa. Chi lam khi co y dinh lock lai.')
else:
    SMOKE = smoke()

---

# Hết phần tiền-GPU — `0a`–`0d` đã xong

| bước | kết quả |
|---|---|
| `0a` manifest | cells `7,430 / 11,191 / 6,617` ≫ 500 · subjects `1,206 / 1,742 / 732` ≫ 200 |
| `0a` timestamp | `0` missing · `1` midnight (**dưới** mức ngẫu nhiên `N/86400 ≈ 2.4`) · `6` dup ⇒ **không loại gì** |
| `0a` concentration | gộp `11.24%` < 15% · per-class `9.50–10.80%` đều dưới ngưỡng |
| `0b` `β_train` | `0.9329 / 0.8596 / 0.8613`, NonTarget-10 `= 1.0` |
| `0c` identity gate | ba tầng `≤ 1e-6` · counter-check giữ CLS **có** lệch |
| `0c` smoke | 6/6 hard-failure pass · `BATCH 128` · `38 h` cho 3 seed |
| `0d` khớp gradient | pooled `1.000294`, CI95 `[0.9934, 1.0076]` **chứa 1.0** |

**GO.** Sang §13 bước 1: lock, rồi bước 2: seed 42.

### Hai điều đọc từ số, không phải từ smoke

**Cỡ can thiệp lấy từ `C8`, không lấy từ smoke.** `C8` trên toàn TRAIN: contested chiếm
`12.82%` gradient âm của Rare-3 ⇒ `R2` xoá `11.5%`, tức **`0.84%`** tổng khối lượng gradient âm
13 nhãn. Smoke cho `8.3%` — đó là nhiễu thành phần mẫu của 13.5% một epoch, không phải cỡ thật.

**Tỉ số `R1GM/R2` bền với lệch thành phần, `R*/R0` thì không.** Vì `β` khớp **theo từng nhãn**,
khi tỉ trọng giữa các nhãn đổi thì cả tử và mẫu đổi cùng nhau — đó là lý do cả hai arm lệch lên
đúng `3.2 pp` như nhau mà tỉ số vẫn `1.000`.

## C10 · **LOCK** (§13 bước 1)

Khoá manifest hash, `β`, batch, và toàn bộ hyperparameter vào `locked_config.json`.

Điểm khiến lock là lock thật, không phải trang trí: **chạy lại cell này sẽ so lại và raise nếu
lệch**. Nếu ai đổi `η`, đổi cửa sổ 7 ngày, đổi `β`, hay manifest bị tính lại khác đi, cell fail
ngay thay vì âm thầm train một estimand khác.

`C12` assert `LOCK_OK` trước khi train.

### Code commit

Colab không có git repo nên `git rev-parse` sẽ fail — đó là bình thường. Cell ghi lại phiên bản
`torch`/`transformers`/GPU và bắt buộc anh điền `CODE_COMMIT` **thủ công** sau khi commit
notebook vào repo. Trường đó để `None` thì lock vẫn dùng được nhưng bản ghi thiếu một mắt.

In [ ]:
import hashlib, platform, datetime
LOCK_JS = f'{SC_DIR}/locked_config.json'
CODE_COMMIT = None          # <<< dien sha sau khi commit notebook vao repo

def sha256_file(p, buf=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(buf)
            if not b: break
            h.update(b)
    return h.hexdigest()

def sha256_obj(o):
    return hashlib.sha256(json.dumps(o, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

try:
    import subprocess
    CODE_COMMIT = CODE_COMMIT or subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                                                text=True, timeout=10).stdout.strip() or None
except Exception:
    pass

_files = {n: f'{SC_DIR}/metrics/{n}' for n in
          ('contested_mask.npy', 'contested_cells.parquet', 'gradient_calibration.json',
           'initial_gradient_share_train_s42.json', 'initial_gradient_share_train_s1337.json')}
_files['cohort_raw_labels.parquet'] = RAW_PQ
HASHES = {n: sha256_file(p) for n, p in _files.items() if os.path.exists(p)}
_miss = [n for n, p in _files.items() if not os.path.exists(p)]
assert not _miss, f'thieu artifact de lock: {_miss}'

import transformers
LOCKED = dict(
    seed=SEED, arms=ARMS, eta=ETA, epochs=EPOCHS, eff=EFF,
    batch=int(BATCH), accum=int(ACCUM),
    rare3=RARE3, worst3_fixed=WORST3_FIXED,
    rare3_idx=RARE3_IDX.tolist(), nontgt_idx=NONTGT_IDX.tolist(), tail_idx=TAIL_IDX.tolist(),
    beta=[float(b) for b in BETA],
    beta_source=dict(transform='train_tf', aug_seeds=list(AUG_SEEDS), workers=int(AUG_WORKERS),
                     precision='FP32', mode='eval()', scope='toan bo TRAIN images',
                     token='patch-only backbone.layernorm(h)[:, 1:]'),
    manifest=dict(window_days=WINDOW_DAYS, anchor='raw +1 only', same_view=REQUIRE_SAME_VIEW,
                  nearest_anchor_per_cell=True, adjacency_required=False, train_only=True,
                  n_contested={l: int(CONTESTED[:, LABELS.index(l)].sum()) for l in RARE3}),
    timestamp_rule=dict(decision='KHONG loai gi',
                        n_missing=int(n_missing), n_midnight=int(_midnight.sum()),
                        n_dup_subject_time=int(n_dup),
                        confirmed_by='nguoi dung xac nhan 00:00:00 la 0 gio dem HOP LE (2026-07-30)',
                        rationale=('1 study o dung 00:00:00 tren %d study la DUOI muc ngau nhien '
                                   'N/86400 ~ %.1f => khong co artifact date-only. 0 missing xac nhan '
                                   'khong co ban ghi thieu StudyTime nen khong the bi dien bu. '
                                   '%d dup subject+time bi d>0 loai tu dong. Nguoi dung xac nhan '
                                   '00:00:00 la gio chup that.' % (N, N/86400.0, n_dup))),
    optim=dict(lr_blocks=LR_BLOCKS, lr_head=LR_HEAD, clip_blocks=CLIP_BLOCKS, clip_head=CLIP_HEAD,
               wd=WD, wd_scope='chi ndim>1', warmup_frac=WARMUP_FRAC, l2sp=LAMBDA_SP,
               head_warmup_epochs=HEAD_WARMUP_EPOCHS,
               sched='tail: warmup+cosine | head: constant (thua tu nb12/13)'),
    asl=dict(gp=ASL_GP, gn=ASL_GN, clip=ASL_CLIP, eps=ASL_EPS,
             norm='sum(w*l)/(B*C) — HANG SO, khong chia sum(w)'),
    arch=dict(train_from=TRAIN_FROM, n_blocks=N_BLOCKS, n_patch=N_PATCH,
              tail='blocks 9-11 + backbone.layernorm, ban sao rieng moi arm',
              head='H5 norm + ML-Decoder, frozen ep1'),
    eval=dict(w_f=W_F, w_l=W_L, macro_tol=MACRO_TOL, n_boot=N_BOOT, cluster='subject_id',
              primary_endpoint='D2 = R2 - R1GM',
              ckpt_rule=f'Macro >= arm best - {MACRO_TOL} -> max Tail AP -> Worst-3 -> Recall@K -> Macro',
              gate=('Rare-3 mAP >= +0.002 OR Macro >= +0.002; NonTarget-10 >= -0.002; '
                    'same direction >= 2/3 seeds; subject-cluster bootstrap no material harm'),
              no_test_access=True),
    claim_locked='gradient matched at the output/logit level at initialization',
    claim_forbidden='parameter-gradient matched throughout training',
    stochastic=bool(STOCHASTIC),
    hashes=HASHES, code_commit=CODE_COMMIT,
    env=dict(torch=torch.__version__, transformers=transformers.__version__,
             python=platform.python_version(),
             gpu=(torch.cuda.get_device_name(0) if dev == 'cuda' else 'cpu')))
_KEYS_STRICT = ['seed','arms','eta','epochs','eff','batch','accum','rare3','worst3_fixed',
                'rare3_idx','nontgt_idx','tail_idx','beta','beta_source','manifest',
                'timestamp_rule','optim','asl','arch','eval','claim_locked','hashes']

if os.path.exists(LOCK_JS):
    OLD = json.load(open(LOCK_JS))
    diff = [k for k in _KEYS_STRICT if sha256_obj(OLD.get(k)) != sha256_obj(LOCKED[k])]
    if diff:
        for k in diff:
            print(f'  LECH [{k}]')
            print(f'    locked : {json.dumps(OLD.get(k), sort_keys=True)[:300]}')
            print(f'    hien   : {json.dumps(LOCKED[k], sort_keys=True)[:300]}')
        raise RuntimeError(f'CAU HINH LECH KHOI LOCK o {len(diff)} muc: {diff}. '
                           f'Hoac sua lai cho khop, hoac doi SC_DIR/SEED va lock moi. '
                           f'KHONG sua {LOCK_JS} de cho khop.')
    LOCK, LOCK_OK = OLD, True
    print(f'LOCK da ton tai va TRUNG KHOP toan bo {len(_KEYS_STRICT)} muc  ->  {LOCK_JS}')
    print(f'  locked_utc {OLD.get("locked_utc")}')
else:
    LOCKED['locked_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
    json.dump(LOCKED, open(LOCK_JS, 'w'), indent=1)
    LOCK, LOCK_OK = LOCKED, True
    print(f'LOCK MOI ghi -> {LOCK_JS}')

print()
print('=== NOI DUNG LOCK ===')
print(f'  seed {LOCK["seed"]} | arms {LOCK["arms"]} | eta {LOCK["eta"]} | '
      f'{LOCK["epochs"]} epoch | batch {LOCK["batch"]} x accum {LOCK["accum"]} = {LOCK["eff"]}')
print(f'  beta (Rare-3): ' + ' | '.join(f'{l} {LOCK["beta"][LABELS.index(l)]:.6f}' for l in RARE3))
print(f'  beta (NonTarget-10): tat ca = '
      f'{set(round(LOCK["beta"][j], 9) for j in LOCK["nontgt_idx"])}')
print(f'  manifest: +/-{LOCK["manifest"]["window_days"]}d, anchor {LOCK["manifest"]["anchor"]}, '
      f'cells {LOCK["manifest"]["n_contested"]}')
print(f'  timestamp: {LOCK["timestamp_rule"]["decision"]}  '
      f'({LOCK["timestamp_rule"]["n_midnight"]} midnight / '
      f'{LOCK["timestamp_rule"]["n_missing"]} missing / '
      f'{LOCK["timestamp_rule"]["n_dup_subject_time"]} dup)')
print(f'    {LOCK["timestamp_rule"]["confirmed_by"]}')
print(f'  claim: "{LOCK["claim_locked"]}"')
print(f'  CAM: "{LOCK["claim_forbidden"]}"')
print()
print('=== HASH ===')
for n, h in LOCK['hashes'].items(): print(f'  {h[:16]}...  {n}')
print()
print(f'  code_commit: {LOCK.get("code_commit")}')
if not LOCK.get('code_commit'):
    print('  ! Colab khong co git repo. Sau khi commit notebook, dien CODE_COMMIT o dau cell')
    print('    roi XOA locked_config.json va lock lai (day la lan duy nhat duoc phep lock lai).')
print(f'  env: torch {LOCK["env"]["torch"]} | transformers {LOCK["env"]["transformers"]} | '
      f'{LOCK["env"]["gpu"]}')
print()
print(f'LOCK_OK = {LOCK_OK}  ->  C12 duoc phep train.')

## C11 · Evaluation (§9)

Pool study-level `frontal 0.7 / lateral 0.3` như nb11–15 — giữ nguyên để so được với mốc cũ.

Endpoint primary (§9): `Macro mAP`, **`Rare-3 mAP`**, **`NonTarget-10 mAP`**, `Tail AP`,
`Worst-3 fixed`, `Recall@K`. Secondary: `AUROC`, `AUPRC`, per-label AP.

Checkpoint rule (§9), **giống nb12–15**:

```
Macro >= arm best Macro - 0.0015  ->  max Tail AP  ->  Worst-3  ->  Recall@K  ->  Macro
```

Chú ý `Tail AP` **không phải** `Rare-3 mAP`: tier `tail` chỉ có `Pleural_Other`, còn Rare-3 có
ba nhãn. Hai đại lượng khác nhau — rule dùng `Tail AP`, còn gate §10 dùng `Rare-3 mAP`.

### Ba điểm hiện thực

**Val loss dùng `w = 1` cho mọi arm.** `IS_BLANK` phủ cả ba split nên `W['R1GM']` có `β` ở cả
val; nếu tính val loss theo weight của arm thì hai arm khác nhau vì một lý do không liên quan
đến những gì chúng học. Loss không-weight so được giữa các arm và cùng thang với train loss.

**Lưu prediction study-level mỗi epoch** (`~1.6 MB/arm/epoch`) để `C13` bootstrap được mà không
phải chạy lại `predict`.

**`persistent_workers=False`** cho eval loader — nguồn của `can only test a child process`.

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, auc
PRED_DIR = f'{SC_DIR}/preds';  SNAP_DIR = f'{SC_DIR}/snapshots'
for _d in (PRED_DIR, SNAP_DIR): os.makedirs(_d, exist_ok=True)

def eval_loader(split, bs, workers=4):
    return DataLoader(ImgDS(IDX[split], eval_tf), batch_size=bs, shuffle=False, collate_fn=collate,
                      num_workers=workers, pin_memory=(dev == 'cuda'), persistent_workers=False)

@torch.no_grad()
def predict(models, split, bs):
    for m in models.values(): m.eval()
    P = {t: np.zeros((len(IDX[split]), C), 'float32') for t in models}
    L = {t: 0.0 for t in models}
    pos, nb = 0, 0
    for b in tqdm(eval_loader(split, bs), desc=f'pred[{split}]', leave=False):
        b = to_dev(b); z = forward_shared(b['image']); k = z.shape[0]
        ones = torch.ones_like(b['y'])
        for t, m in models.items():
            zz = m(z)
            P[t][pos:pos+k] = torch.sigmoid(zz).float().cpu().numpy()
            L[t] += float(CRIT(zz, b['y'], ones))       # w=1: so duoc giua cac arm
        pos += k; nb += 1
    return P, {t: L[t]/max(nb, 1) for t in L}

def pool_study(p_img, split):
    sub = man.iloc[IDX[split]]
    d = pd.DataFrame(p_img, columns=LABELS)
    d['study_id'] = sub.study_id.to_numpy(); d['vg'] = sub.view_group.to_numpy()
    sid = np.sort(d.study_id.unique())
    fr = d[d.vg == 'frontal'].groupby('study_id')[LABELS].mean().reindex(sid)
    la = d[d.vg == 'lateral'].groupby('study_id')[LABELS].mean().reindex(sid)
    hf = fr.notna().all(1).to_numpy().astype('float32'); hl = la.notna().all(1).to_numpy().astype('float32')
    den = (W_F*hf + W_L*hl)[:, None]
    return sid, (W_F*np.nan_to_num(fr.to_numpy('float32'))*hf[:, None]
                 + W_L*np.nan_to_num(la.to_numpy('float32'))*hl[:, None]) / den

def recall_at_k(y, s):
    K = int(y.sum())
    if K == 0: return np.nan
    return float(y[np.argsort(-s, kind='stable')[:K]].sum())/K

def metrics_from(Y, PS):
    ap = np.array([average_precision_score(Y[:, j], PS[:, j]) if Y[:, j].sum() else np.nan
                   for j in range(C)])
    auprc = np.full(C, np.nan); auroc = np.full(C, np.nan); rk = np.full(C, np.nan)
    for j in range(C):
        if Y[:, j].sum() == 0 or Y[:, j].sum() == len(Y): continue
        pr, rc, _ = precision_recall_curve(Y[:, j], PS[:, j])
        auprc[j] = auc(rc, pr); auroc[j] = roc_auc_score(Y[:, j], PS[:, j])
        rk[j] = recall_at_k(Y[:, j], PS[:, j])
    nm = lambda a, ix: float(np.nanmean(a[ix]))
    return dict(macro_map=float(np.nanmean(ap)),
                rare3_map=nm(ap, RARE3_IDX), nontgt_map=nm(ap, NONTGT_IDX),
                tail_map=nm(ap, TAIL_IDX), head_map=nm(ap, TIER == 'head'),
                med_map=nm(ap, TIER == 'medium'), worst3_fixed=nm(ap, W3_IDX),
                tail_recall_k=nm(rk, TAIL_IDX), rare3_recall_k=nm(rk, RARE3_IDX),
                macro_auprc=float(np.nanmean(auprc)), macro_auroc=float(np.nanmean(auroc)),
                rare3_auroc=nm(auroc, RARE3_IDX), tail_auroc=nm(auroc, TAIL_IDX),
                per_ap=ap.tolist(), per_auprc=auprc.tolist(),
                per_auroc=auroc.tolist(), per_rk=rk.tolist())

SID_VAL, _ = pool_study(np.zeros((len(IDX['val']), C), 'float32'), 'val')
Y_VAL = ehr.set_index('study_id').loc[SID_VAL, LABELS].to_numpy('float32')
SUBJ_VAL = ehr.set_index('study_id').loc[SID_VAL, 'subject_id'].to_numpy()
np.save(f'{PRED_DIR}/sid_val.npy', SID_VAL)
np.save(f'{PRED_DIR}/y_val.npy', Y_VAL)
np.save(f'{PRED_DIR}/subj_val.npy', SUBJ_VAL)
print(f'VAL: {len(SID_VAL):,} study | {len(np.unique(SUBJ_VAL)):,} subject | '
      f'{len(IDX["val"]):,} anh')
print(f'  positives/nhan (Rare-3): '
      + ' | '.join(f'{l} {int(Y_VAL[:, LABELS.index(l)].sum())}' for l in RARE3))
print('  TEST khong bao gio duoc doc.')

def select_ckpt(log_arm):
    """§9: Macro >= best - MACRO_TOL -> max Tail AP -> Worst-3 -> Recall@K -> Macro."""
    e = {int(k): v for k, v in log_arm.items()}
    best = max(v['macro_map'] for v in e.values())
    cand = [k for k, v in e.items() if v['macro_map'] >= best - MACRO_TOL]
    return max(cand, key=lambda k: (e[k]['tail_map'], e[k]['worst3_fixed'],
                                    e[k]['tail_recall_k'], e[k]['macro_map']))

# ---- probe batch co dinh cho feature diagnostics (§7.3) ----
PROBE = to_dev(collate([ImgDS(IDX['val'], eval_tf)[k] for k in range(16)]))
Z_PROBE = forward_shared(PROBE['image'])

@torch.no_grad()
def feat_stats(m, f0=None):
    was = m.training; m.eval()
    f = m.tail(Z_PROBE)[:, 1:].mean(1).float()               # patch-only, pooled
    m.train(was)
    fs = f - f.mean(0, keepdim=True)
    ev = torch.linalg.svdvals(fs)**2
    out = dict(feat_var=float(f.var(0).mean()),
               feat_rank_pr=float((ev.sum()**2)/(ev**2).sum().clamp(min=1e-30)))
    if f0 is not None:
        out['feat_cos_init'] = float(F.cosine_similarity(f, f0, dim=1).mean())
    return out, f

def rel_drift(sd, init, prefix):
    """init duoc luu tren CPU (tiet kiem VRAM), sd tren CUDA -> phai dua ve cung device."""
    ks = [k for k in sd if k.startswith(prefix)]
    if not ks: return 0.0
    num = den = 0.0
    for k in ks:
        a = sd[k].detach().float().cpu(); b = init[k].detach().float().cpu()
        num += float(((a-b)**2).sum()); den += float((b**2).sum())
    return (num**0.5)/max(den**0.5, 1e-12)
DRIFT_GROUPS = {'b9': 'tail.blocks.0.', 'b10': 'tail.blocks.1.', 'b11': 'tail.blocks.2.',
                'bb_ln': 'tail.ln.', 'h5_norm': 'head.norm.', 'h5_mld': 'head.mld.'}

# ---- §3.2: parameter-gradient ratio + cosine R2 vs R1GM (DIAGNOSTIC) ----
GRAD_GROUPS = {'tail': lambda m: tail_params(m), 'bb_ln': lambda m: list(m.tail.ln.parameters()),
               'head': lambda m: head_params(m)}
def grad_stats(ST):
    """Goi TRUOC clip. Tra norm tung group tung arm + cos(R2, R1GM)."""
    out = {}
    for gname, sel in GRAD_GROUPS.items():
        gs = {a: [p.grad for p in sel(ST[a]['m']) if p.grad is not None] for a in ST}
        for a in ST:
            out[f'gn_{gname}_{a}'] = float(torch.sqrt(sum((g.float()**2).sum() for g in gs[a]))) \
                                    if gs[a] else 0.0
        if 'R2' in gs and 'R1GM' in gs and gs['R2'] and gs['R1GM']:
            dot = float(sum((x.float()*y.float()).sum() for x, y in zip(gs['R2'], gs['R1GM'])))
            n2 = out[f'gn_{gname}_R2']; n1 = out[f'gn_{gname}_R1GM']
            out[f'cos_{gname}_R2_R1GM'] = dot/max(n2*n1, 1e-30)
            out[f'ratio_{gname}_R2_R1GM'] = n2/max(n1, 1e-30)
    return out
print('eval + diagnostics ready | checkpoint rule:', LOCK['eval']['ckpt_rule'])

## C12 · Train seed 42 (§13 bước 2) — 8 epoch × 3 arm

Ba arm khởi tạo **bit-for-bit giống nhau**, cùng thứ tự batch, cùng ảnh tăng cường (vì `b` tính
một lần rồi chia sẻ), cùng dropout mask nếu `STOCHASTIC`. Khác biệt duy nhất là weight matrix.

| | |
|---|---|
| ep1 | tail (blocks 9–11 + `backbone.layernorm`) train · H5 `norm`+`mld` **frozen** |
| ep2–8 | tất cả train |
| LR | tail warmup+cosine · head **constant** (thừa từ nb12/13 — tránh việc head vừa mở đã bị cosine kéo xuống) |
| clip | tail `5.0` · head `1.0` |
| resume | mỗi epoch, có guard theo `batch` **và** hash của lock |

Log mỗi epoch (§7.3): `relative parameter drift` từng block, `feature cosine/variance/rank`,
`train/val gap`, và (§3.2) `parameter-gradient ratio R2/R1GM` + `cosine R2·R1GM` theo nhóm.

Tất cả những cái đó là **DIAGNOSTIC**. Hard failure chỉ 6 mục ở `C9`. Riêng §3.3 nói trước:
`R2/R1GM` gradient ratio **được phép tăng** theo epoch — `R2` giảm áp lực lên contested
high-score blanks nên điểm nhóm đó tăng, kéo raw ASL negative gradient tăng. **Đó là động lực
điều trị mong đợi, không phải lỗi.** Không đổi weight động.

**Chi phí:** `~1.57 h/epoch` → **`~12.5 h`** cho seed này. Vừa session Colab Pro+ 24 h. Resume
theo epoch tốn `<0.5%` thời gian epoch — giữ, vì mất kết nối ở giờ thứ 11 thì mất cả 11 giờ.

In [ ]:
assert LOCK_OK, 'chua lock -> chay C10 truoc'
assert SEED == LOCK['seed'] and int(BATCH) == LOCK['batch'], 'SEED/BATCH lech khoi lock'
assert np.allclose(BETA, LOCK['beta']), 'BETA lech khoi lock'
CONT_T = torch.from_numpy(CONTESTED.astype('float32')).to(dev)   # HOIST: dung tao lai moi step
LOG_PATH  = f'{SC_DIR}/metrics/epoch_log_seed{SEED}.json'
resume_p  = f'{SC_DIR}/checkpoints/_resume_seed{SEED}.pt'
snap_p    = lambda a, ep: f'{SNAP_DIR}/{a}_s{SEED}_ep{ep}.pt'
pred_p    = lambda a, ep: f'{PRED_DIR}/{a}_s{SEED}_ep{ep}.npy'
LOCK_SIG  = sha256_obj({k: LOCK[k] for k in _KEYS_STRICT})

def train_seed(batch=BATCH, epochs=EPOCHS):
    dl = loader('train', train_tf, True, batch, workers=4)
    accum = ACCUM
    STEPS = math.ceil(len(dl)/accum); TOTAL = epochs*STEPS; WARM = max(1, int(WARMUP_FRAC*TOTAL))

    def mk_sched(opt):
        cw = lambda s: s/WARM if s < WARM else 0.5*(1+math.cos(math.pi*(s-WARM)/max(TOTAL-WARM, 1)))
        const = lambda s: 1.0
        return torch.optim.lr_scheduler.LambdaLR(opt, [cw, cw, const, const])   # tail x2, head x2

    LOG = json.load(open(LOG_PATH)) if os.path.exists(LOG_PATH) else {}
    ST, start = {}, 0
    if os.path.exists(resume_p):
        ck = torch.load(resume_p, map_location=dev, weights_only=False)
        if ck.get('batch') != batch:
            raise RuntimeError(f"resume batch {ck.get('batch')} != {batch}; xoa {resume_p}")
        if ck.get('lock_sig') != LOCK_SIG:
            raise RuntimeError('resume duoc tao voi mot LOCK KHAC. Khong duoc tiep tuc: '
                               'ket qua se tron hai cau hinh. Xoa resume hoac doi SC_DIR.')
        for a in ARMS:
            m = Arm().to(dev); m.load_state_dict(ck[a]['model'])
            o = make_opt(m); o.load_state_dict(ck[a]['opt'])
            s = mk_sched(o); s.load_state_dict(ck[a]['sch'])
            ST[a] = dict(m=m, opt=o, sch=s, init=ck[a]['init'], f0=ck[a]['f0'].to(dev))
        start = ck['epoch'] + 1
        print(f'RESUMED seed {SEED} @ep{start+1}')
        if start >= epochs:
            print('da chay xong -> sang C13'); return LOG
    else:
        base = build_arm(SEED)
        for a in ARMS:
            m = Arm().to(dev)
            m.load_state_dict({k: v.detach().clone() for k, v in base.state_dict().items()})
            o = make_opt(m)
            _, f0 = feat_stats(m)
            ST[a] = dict(m=m, opt=o, sch=mk_sched(o),
                         init={k: v.detach().cpu().clone() for k, v in m.state_dict().items()}, f0=f0)
        _sd = [ST[a]['m'].state_dict() for a in ARMS]
        assert all(torch.equal(x, y) for d in _sd[1:] for x, y in zip(_sd[0].values(), d.values())), \
            'cac arm KHONG cung init -> khong con paired'
        del base
        print(f'init: {len(ARMS)} arm bit-for-bit | {STEPS:,} opt-step/epoch | total {TOTAL:,} '
              f'| warmup {WARM}')

    FROZEN0 = {k: v.detach().clone() for k, v in backbone.state_dict().items()}
    for ep in range(start, epochs):
        unf = ep >= HEAD_WARMUP_EPOCHS
        tag = '[all]' if unf else '[tail]'
        for a in ARMS: set_head_trainable(ST[a]['m'], unf); ST[a]['m'].train()
        acc = {a: dict(loss=0.0, n=0, gm=0.0, gm_c=0.0, gs=[], nopt=0) for a in ARMS}
        torch.manual_seed(SEED*7919 + ep)                     # thu tu batch, on dinh qua resume
        t0 = time.time()
        for i, b in enumerate(tqdm(dl, desc=f's{SEED} ep{ep+1}/{epochs}{tag}', leave=False)):
            b = to_dev(b); z = forward_shared(b['image'])
            step_now = ((i+1) % accum == 0) or (i == len(dl)-1)
            for a in ARMS:
                e = ST[a]
                if STOCHASTIC: sync_rng(ep, i)
                zz = e['m'](z)
                loss = CRIT(zz, b['y'], Wt[a][b['sr']])
                assert torch.isfinite(loss), f'HARD FAILURE: loss non-finite {a} ep{ep+1} step{i}'
                (loss/accum).backward()
                d = acc[a]; d['loss'] += float(loss.detach()); d['n'] += 1
                with torch.no_grad():
                    _g = asl_neg_grad(zz)*(1-b['y'])*Wt[a][b['sr']]
                    d['gm'] += float((_g*BLANK_T[b['sr']])[:, RARE3_IDX].sum())
                    d['gm_c'] += float((_g*CONT_T[b['sr']])[:, RARE3_IDX].sum())
            if step_now:
                if acc[ARMS[0]]['nopt'] % 20 == 0:            # §3.2 diagnostic, thua thot
                    gsx = grad_stats(ST)
                    for a in ARMS: acc[a]['gs'].append(gsx)
                for a in ARMS:
                    nt, nh = clip_arm(ST[a]['m'])
                    assert math.isfinite(nt) and math.isfinite(nh), \
                        f'HARD FAILURE: grad norm non-finite {a} ep{ep+1} step{i}'
                    ST[a]['opt'].step(); ST[a]['sch'].step()
                    ST[a]['opt'].zero_grad(set_to_none=True); acc[a]['nopt'] += 1
        h_tr = (time.time()-t0)/3600

        _bad = [k for k, v in backbone.state_dict().items() if not torch.equal(FROZEN0[k], v)]
        assert not _bad, f'HARD FAILURE: shared blocks 0-8 bi doi: {_bad[:5]}'

        Pv, Lv = predict({a: ST[a]['m'] for a in ARMS}, 'val', batch)
        rows = []
        for a in ARMS:
            sid, PS = pool_study(Pv[a], 'val')
            assert (sid == SID_VAL).all()
            np.save(pred_p(a, ep+1), PS)
            r = metrics_from(Y_VAL, PS)
            sd = ST[a]['m'].state_dict(); ini = ST[a]['init']
            fs, _ = feat_stats(ST[a]['m'], ST[a]['f0'])
            d = acc[a]; n = max(d['n'], 1)
            gs = {k: float(np.mean([g[k] for g in d['gs']])) for k in (d['gs'][0] if d['gs'] else {})}
            r['diag'] = dict(train_loss=d['loss']/n, val_loss=Lv[a],
                             train_val_gap=Lv[a]-d['loss']/n, hours_train=h_tr,
                             head_unfrozen=bool(unf), lr_tail=ST[a]['opt'].param_groups[0]['lr'],
                             lr_head=ST[a]['opt'].param_groups[2]['lr'],
                             grad_mass_blank_rare3=d['gm']/n, grad_mass_contested_rare3=d['gm_c']/n,
                             drift={g: rel_drift(sd, ini, p) for g, p in DRIFT_GROUPS.items()},
                             **fs, **gs)
            LOG.setdefault(a, {})[str(ep+1)] = r
            torch.save({k: v.detach().cpu() for k, v in sd.items()}, snap_p(a, ep+1))
            rows.append(f"{a} mAP {r['macro_map']:.4f} R3 {r['rare3_map']:.4f} "
                        f"NT {r['nontgt_map']:.4f} tail {r['tail_map']:.4f} "
                        f"w3 {r['worst3_fixed']:.4f} L {r['diag']['train_loss']:.4f}")
        json.dump(LOG, open(LOG_PATH, 'w'), indent=1)
        torch.save({**{a: dict(model=ST[a]['m'].state_dict(), opt=ST[a]['opt'].state_dict(),
                               sch=ST[a]['sch'].state_dict(), init=ST[a]['init'],
                               f0=ST[a]['f0'].cpu()) for a in ARMS},
                    'epoch': ep, 'batch': batch, 'lock_sig': LOCK_SIG}, resume_p+'.tmp')
        os.replace(resume_p+'.tmp', resume_p)
        print(f'  ep{ep+1}{tag} {h_tr:.2f}h | ' + ' | '.join(rows) + '  [saved]')
        a0 = LOG[ARMS[0]][str(ep+1)]
        for a in ARMS[1:]:
            v = LOG[a][str(ep+1)]
            print(f'    vs {ARMS[0]}: {a:<5} dMacro {v["macro_map"]-a0["macro_map"]:+.4f} '
                  f'dRare3 {v["rare3_map"]-a0["rare3_map"]:+.4f} '
                  f'dNT {v["nontgt_map"]-a0["nontgt_map"]:+.4f} '
                  f'dTail {v["tail_map"]-a0["tail_map"]:+.4f}')
    print()
    print(f'DONE seed {SEED}: {epochs} epoch x {len(ARMS)} arm -> {LOG_PATH}')
    return LOG

LOG = train_seed()

## C13 · `D1` / `D2` + subject-cluster bootstrap (§9, §10)

$$D_1 = R1\text{-}GM - R0 \qquad\text{(hạ áp lực âm chung)}$$
$$D_2 = R2 - R1\text{-}GM \qquad\text{(primary estimand: vị trí, với cùng khối lượng gradient ban đầu)}$$

Bootstrap §9: **2,000 resample, cluster = `subject_id`**. Resample theo bệnh nhân chứ không theo
study — nhiều study của cùng người không độc lập, bootstrap theo study sẽ cho CI **hẹp giả**.

Resample nào có nhãn không còn positive thì AP nhãn đó không định nghĩa được. §9 đòi *"handle
no-positive resamples explicitly and report valid fraction"* — cell báo `valid_frac` từng nhãn,
không âm thầm bỏ.

### Gate §10 cho `D2`

```
Rare-3 mAP >= +0.002  HOAC  Macro mAP >= +0.002
NonTarget-10 >= -0.002
cung chieu >= 2/3 seed
subject-cluster bootstrap khong co material harm
data sufficiency da pass
```

Seed này **một mình không kết luận được** — điều kiện 2/3 seed cần cả `1337` và `2024`. Và theo
§11 phải chạy cả hai **bất kể** dấu của seed 42.

### Bảng diễn giải (§10 + ba dòng thêm ở Q4)

| `D1` | `D2` | đọc thế nào |
|---|---|---|
| `> 0` | `≈ 0` | hạ áp lực âm Rare-3 **chung** là đủ; **vị trí không thêm gì** |
| `> 0` | `> 0` | vị trí theo thời gian **thêm giá trị** ngoài cùng khối lượng gradient ban đầu |
| `> 0` sớm rồi sụp | | rebalance mạnh gây overfit — trùng hành vi nb12 (WASL .50 tụt `0.0132`) |
| `≈ 0` | `≈ 0` | **không** phát hiện được gì ở `η = 0.10`; can thiệp `0.84%` gradient là quá nhỏ |
| `< 0` | `≈ 0` | hạ áp lực âm **có hại**; nhãn blank mang tín hiệu âm thật |
| `≈ 0` | `> 0` | **mạnh nhất cho giả thuyết**: vị trí có giá trị *mà không cần* hạ áp lực âm chung |

Đọc `NonTarget-10` như **placebo**: 10 nhãn đó có `β = 1` ở cả ba arm nên weight y hệt nhau. Nếu
chúng cũng dịch đáng kể thì có gì rò rỉ, và `D2` trên Rare-3 không đáng tin.

In [ ]:
LOG = json.load(open(LOG_PATH))
KEYS = ['macro_map', 'rare3_map', 'nontgt_map', 'tail_map', 'worst3_fixed', 'tail_recall_k',
        'macro_auroc', 'rare3_auroc', 'macro_auprc']
EPS_ = sorted({int(e) for a in ARMS for e in LOG.get(a, {})})

print('=== PER-EPOCH, PAIRED (bang chung so cap) ===')
T = pd.DataFrame([dict(ep=e, arm=a, **{k: LOG[a][str(e)][k] for k in KEYS[:6]},
                       train_loss=LOG[a][str(e)]['diag']['train_loss'],
                       val_loss=LOG[a][str(e)]['diag']['val_loss'],
                       gap=LOG[a][str(e)]['diag']['train_val_gap'])
                  for e in EPS_ for a in ARMS if str(e) in LOG.get(a, {})])
print(T.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
T.to_csv(f'{SC_DIR}/metrics/per_epoch_seed{SEED}.csv', index=False)

print()
print('=== DELTA TUNG EPOCH ===')
D = pd.DataFrame([dict(ep=e, **{f'D1_{k}': LOG['R1GM'][str(e)][k]-LOG['R0'][str(e)][k] for k in KEYS[:3]},
                       **{f'D2_{k}': LOG['R2'][str(e)][k]-LOG['R1GM'][str(e)][k] for k in KEYS[:3]})
                  for e in EPS_ if all(str(e) in LOG.get(a, {}) for a in ARMS)])
print(D.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
D.to_csv(f'{SC_DIR}/metrics/deltas_seed{SEED}.csv', index=False)

print()
print('=== §3.2 / §3.3 DIAGNOSTIC: gradient ratio R2/R1GM theo epoch ===')
G = pd.DataFrame([dict(ep=e, **{k: v for k, v in LOG['R2'][str(e)]['diag'].items()
                                if k.startswith(('ratio_', 'cos_'))},
                       gm_blank_R0=LOG['R0'][str(e)]['diag']['grad_mass_blank_rare3'],
                       gm_blank_R1GM=LOG['R1GM'][str(e)]['diag']['grad_mass_blank_rare3'],
                       gm_blank_R2=LOG['R2'][str(e)]['diag']['grad_mass_blank_rare3'])
                  for e in EPS_ if str(e) in LOG.get('R2', {})])
if len(G):
    G['gm_R1GM_over_R2'] = G.gm_blank_R1GM/G.gm_blank_R2.clip(lower=1e-30)
    print(G.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    print('  §3.3: ratio TANG theo epoch la dong luc dieu tri MONG DOI, khong phai loi.')

print()
print('=== §7.3 DIAGNOSTIC: drift / feature (KHONG dung de gate) ===')
V = pd.DataFrame([dict(ep=e, arm=a, **LOG[a][str(e)]['diag']['drift'],
                       feat_cos=LOG[a][str(e)]['diag'].get('feat_cos_init'),
                       feat_var=LOG[a][str(e)]['diag'].get('feat_var'),
                       rank_pr=LOG[a][str(e)]['diag'].get('feat_rank_pr'))
                  for e in EPS_ for a in ARMS if str(e) in LOG.get(a, {})])
print(V.to_string(index=False, float_format=lambda v: f'{v:.6g}'))

print()
print(f'=== CHECKPOINT theo §9 rule ===')
SEL = {a: select_ckpt(LOG[a]) for a in ARMS if LOG.get(a)}
for a in ARMS:
    r = LOG[a][str(SEL[a])]
    print(f'  {a:<5} ep{SEL[a]} | Macro {r["macro_map"]:.4f} Rare3 {r["rare3_map"]:.4f} '
          f'NT {r["nontgt_map"]:.4f} Tail {r["tail_map"]:.4f} W3 {r["worst3_fixed"]:.4f} '
          f'Rk {r["tail_recall_k"]:.4f}')
print(f'  (rule dung Tail AP; gate §10 dung Rare-3 mAP — hai dai luong KHAC nhau)')

# ---------------- subject-cluster bootstrap ----------------
SUBJ_U = np.unique(SUBJ_VAL)
SUBJ_POS = {s: np.where(SUBJ_VAL == s)[0] for s in SUBJ_U}
_rng = np.random.default_rng(LOCK['seed'])
BOOT_IX = [np.concatenate([SUBJ_POS[s] for s in _rng.choice(SUBJ_U, len(SUBJ_U), replace=True)])
           for _ in range(N_BOOT)]
print()
print(f'bootstrap: {N_BOOT} resample, cluster = subject_id ({len(SUBJ_U):,} subject)')

def ap_vec(Y, P):
    return np.array([average_generic(Y[:, j], P[:, j]) for j in range(C)])
def average_generic(y, p):
    return average_precision_score(y, p) if y.sum() > 0 else np.nan

def boot_deltas(ep_by_arm):
    P = {a: np.load(pred_p(a, ep_by_arm[a])) for a in ARMS}
    grp = dict(rare3=RARE3_IDX, nontgt=NONTGT_IDX, macro=np.arange(C), tail=TAIL_IDX)
    acc = {f'{d}_{g}': [] for d in ('D1', 'D2') for g in grp}
    valid = np.zeros(C); tot = 0
    for ix in BOOT_IX:
        Yb = Y_VAL[ix]
        aps = {a: ap_vec(Yb, P[a][ix]) for a in ARMS}
        valid += ~np.isnan(aps[ARMS[0]]); tot += 1
        for g, jj in grp.items():
            d1 = np.nanmean(aps['R1GM'][jj]) - np.nanmean(aps['R0'][jj])
            d2 = np.nanmean(aps['R2'][jj]) - np.nanmean(aps['R1GM'][jj])
            acc[f'D1_{g}'].append(d1); acc[f'D2_{g}'].append(d2)
    pt = {}
    for a in ARMS:
        ap = ap_vec(Y_VAL, P[a])
        for g, jj in grp.items(): pt[f'{a}_{g}'] = float(np.nanmean(ap[jj]))
    rows = []
    for k, v in acc.items():
        v = np.asarray(v); d, g = k.split('_')
        obs = (pt[f'R1GM_{g}']-pt[f'R0_{g}']) if d == 'D1' else (pt[f'R2_{g}']-pt[f'R1GM_{g}'])
        lo, hi = np.percentile(v, [2.5, 97.5])
        rows.append(dict(delta=d, group=g, observed=obs, boot_mean=float(v.mean()),
                         ci_lo=float(lo), ci_hi=float(hi),
                         p_gt0=float((v > 0).mean()), excludes_0=bool(lo > 0 or hi < 0)))
    return pd.DataFrame(rows), pd.Series(valid/max(tot, 1), index=LABELS)

print()
print('--- (a) checkpoint tung arm theo rule §9 ---')
B_sel, VF = boot_deltas(SEL)
print(B_sel.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
print()
print('--- (b) EPOCH CO DINH = ep8 (khong co selection bias) ---')
B_fix, _ = boot_deltas({a: max(EPS_) for a in ARMS})
print(B_fix.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
print()
print('=== §9: valid fraction cua AP tung nhan qua bootstrap ===')
print(VF.to_string(float_format=lambda v: f'{v:.4f}'))
_lowvf = VF[VF < 0.99]
if len(_lowvf):
    print(f'  ! nhan co resample thieu positive: {dict(_lowvf.round(4))}')
    print('    AP cua chung la nanmean tren tap con -> CI rong hon va hoi lech. Ghi vao bao cao.')
else:
    print('  moi nhan co positive trong >=99% resample.')

# ---------------- GATE §10 ----------------
def gate(B, tag):
    g = lambda d, grp: B[(B.delta == d) & (B.group == grp)].iloc[0]
    d2r, d2m, d2n = g('D2', 'rare3'), g('D2', 'macro'), g('D2', 'nontgt')
    d1r = g('D1', 'rare3')
    c1 = (d2r.observed >= 0.002) or (d2m.observed >= 0.002)
    c2 = d2n.observed >= -0.002
    c4 = not (d2r.ci_hi < -0.002 or d2n.ci_hi < -0.002)
    print()
    print(f'=== GATE §10 cho D2 [{tag}] ===')
    print(f'  Rare-3 >= +0.002 HOAC Macro >= +0.002 : Rare3 {d2r.observed:+.4f} / '
          f'Macro {d2m.observed:+.4f}  -> {"PASS" if c1 else "FAIL"}')
    print(f'  NonTarget-10 >= -0.002                : {d2n.observed:+.4f}  '
          f'-> {"PASS" if c2 else "FAIL"}')
    print(f'  bootstrap khong material harm         : CI(D2 Rare3) [{d2r.ci_lo:+.4f}, {d2r.ci_hi:+.4f}], '
          f'CI(D2 NT) [{d2n.ci_lo:+.4f}, {d2n.ci_hi:+.4f}]  -> {"PASS" if c4 else "FAIL"}')
    print(f'  data sufficiency                      : PASS (C4 + C8)')
    print(f'  cung chieu >= 2/3 seed                : CHUA XAC DINH (can seed 1337 + 2024)')
    print(f'  tham chieu D1 Rare-3                  : {d1r.observed:+.4f} '
          f'CI [{d1r.ci_lo:+.4f}, {d1r.ci_hi:+.4f}]')
    print()
    _t = 0.002
    d1v, d2v = d1r.observed, d2r.observed
    if abs(d1v) < _t and abs(d2v) < _t:
        it = 'D1~0 & D2~0 -> KHONG phat hien duoc gi tai eta=0.10. Can thiep 0.84% gradient la qua nho.'
    elif d1v >= _t and abs(d2v) < _t:
        it = 'D1>0 & D2~0 -> ha ap luc am Rare-3 CHUNG la du; vi tri theo thoi gian khong them gi.'
    elif d2v >= _t and abs(d1v) < _t:
        it = 'D1~0 & D2>0 -> MANH NHAT cho gia thuyet: vi tri co gia tri MA KHONG can ha ap luc am chung.'
    elif d1v >= _t and d2v >= _t:
        it = 'D1>0 & D2>0 -> vi tri theo thoi gian them gia tri ngoai cung khoi luong gradient ban dau.'
    elif d1v <= -_t:
        it = 'D1<0 -> ha ap luc am CO HAI; o blank mang tin hieu am that.'
    else:
        it = 'khong khop dong nao trong bang -> mo ta bang so, KHONG dat ten ket cuc moi.'
    print(f'  DIEN GIAI (§10): {it}')
    return dict(tag=tag, c_effect=bool(c1), c_nontgt=bool(c2), c_boot=bool(c4),
                d1_rare3=float(d1v), d2_rare3=float(d2v), d2_macro=float(d2m.observed),
                d2_nontgt=float(d2n.observed), interpretation=it)

RES = dict(seed=SEED, lock_sig=LOCK_SIG, selected_epoch=SEL,
           gate_selected=gate(B_sel, 'checkpoint §9'), gate_fixed_ep8=gate(B_fix, 'ep8 co dinh'),
           valid_fraction={l: float(v) for l, v in VF.items()},
           boot_selected=B_sel.to_dict('records'), boot_fixed_ep8=B_fix.to_dict('records'))
json.dump(RES, open(f'{SC_DIR}/metrics/result_seed{SEED}.json', 'w'), indent=1)
B_sel.to_csv(f'{SC_DIR}/metrics/bootstrap_seed{SEED}_selected.csv', index=False)
B_fix.to_csv(f'{SC_DIR}/metrics/bootstrap_seed{SEED}_ep8.csv', index=False)
print()
print(f'-> {SC_DIR}/metrics/result_seed{SEED}.json')
print()
print('=== BUOC TIEP (§11 — DOC KY) ===')
print('  Chay seed 1337 va 2024 BAT KE dau cua delta seed nay.')
print('  Doi SEED o C0 -> chay lai C10 (lock se so khop, chi `seed` duoc phep khac) -> C12 -> C13.')
print('  Dung sau seed 42 CHI vi: implementation failure / data-sufficiency failure /')
print('  numerical instability / gross predefined safety harm.')
print('  Neu dung vi futility hieu nang: bao cao one-seed exploratory NO-GO, KHONG claim duong.')

---

## C14 · Diagnostic — `ep0` exact init · optimization trace · AP từng nhãn

Cell **chỉ đọc**: không ghi đè log, không đụng lock, không đổi state của `C12`. Chỉ thêm hai file
mới (`EP0_s{SEED}.npy`, `diag_seed{SEED}.json`).

Ba kiểm tra, dựng để trả lời một câu duy nhất: **epoch 1 (head frozen) có làm hại không.**

### A · `ep0` = init chính xác

Không dựng lại `Arm()` rồi hy vọng nó trùng — **đọc thẳng `init` của `R0` từ file resume**, tức
đúng trọng số mà `C12` khởi tạo. Kèm assert ba arm có `init` **bit-for-bit** giống nhau.

Đây là mốc chưa từng có: `04b` backbone + `H5` head patch-only, **chưa train gì**. So với `ep1`
cho biết epoch 1 đã thêm hay bớt.

| `ep0` vs `ep1` | đọc thế nào |
|---|---|
| `ep1` > `ep0` rõ | epoch 1 có ích ⇒ head-freeze không gây hại đáng kể |
| `ep1` ≈ `ep0` | epoch 1 vô ích — 1.4 h GPU không đổi gì |
| `ep1` < `ep0` | **epoch 1 làm hại** ⇒ giả thuyết representation–head mismatch được ủng hộ |

### B · Optimization trace

`val_loss`, `train_val_gap`, `lr_tail`/`lr_head`, drift từng block, `feat_cos_init`. Tất cả đã
có sẵn trong `epoch_log_seed{SEED}.json`, chỉ cần đọc ra.

Kèm **chiếu LR toàn bộ 8 epoch** — tính từ công thức, không cần dữ liệu — để thấy rõ tỉ số
`lr_head / lr_tail` lớn dần tới mức nào ở các epoch cuối.

### C · AP từng nhãn + `D1`/`D2` từng nhãn

Kiểm giả thuyết: **`D2` theo nhãn có đi cùng chiều với `enrich` không.** `Pleural_Other` enrich
`3.40×`, `Lung_Lesion` `1.99×`, `Fracture` `1.45×`. Nếu `D2` xếp cùng thứ tự thì đó là bằng
chứng **cơ chế**, kể cả khi cổng tổng hợp fail.

Không suy từ số gộp nữa — `per_ap` có sẵn trong log.

In [ ]:
DIAG_JS = f'{SC_DIR}/metrics/diag_seed{SEED}.json'
EP0_NPY = f'{PRED_DIR}/EP0_s{SEED}.npy'
LOG = json.load(open(LOG_PATH))
EPS_ = sorted({int(e) for e in LOG.get(ARMS[0], {})})
print(f'log co epoch: {EPS_}')

# ================= A · ep0 = init CHINH XAC =================
if os.path.exists(EP0_NPY):
    PS0 = np.load(EP0_NPY); L0v = json.load(open(f'{SC_DIR}/metrics/ep0_loss_s{SEED}.json'))['val_loss']
    print(f'ep0 <- cache {os.path.basename(EP0_NPY)}')
else:
    assert os.path.exists(resume_p), 'khong co file resume -> khong lay duoc init chinh xac'
    _ck = torch.load(resume_p, map_location='cpu', weights_only=False)
    _i0 = _ck[ARMS[0]]['init']
    for a in ARMS[1:]:
        _bad = [k for k in _i0 if not torch.equal(_i0[k], _ck[a]['init'][k])]
        assert not _bad, f'init cua {a} KHAC {ARMS[0]} tai {_bad[:3]} -> khong con paired'
    print(f'{len(ARMS)} arm co init bit-for-bit giong nhau  OK')
    m0 = Arm().to(dev); m0.load_state_dict({k: v.to(dev) for k, v in _i0.items()})
    P0, L0 = predict({'EP0': m0}, 'val', BATCH)
    sid0, PS0 = pool_study(P0['EP0'], 'val'); assert (sid0 == SID_VAL).all()
    L0v = L0['EP0']
    np.save(EP0_NPY, PS0)
    json.dump(dict(val_loss=L0v), open(f'{SC_DIR}/metrics/ep0_loss_s{SEED}.json', 'w'))
    del m0, _ck; torch.cuda.empty_cache() if dev == 'cuda' else None
M0 = metrics_from(Y_VAL, PS0)

print()
print('=== A · ep0 (04b backbone + H5 head patch-only, CHUA TRAIN) vs cac epoch ===')
KEYS_A = ['macro_map', 'rare3_map', 'nontgt_map', 'tail_map', 'worst3_fixed', 'tail_recall_k']
rows = [dict(ep=0, arm='INIT', **{k: M0[k] for k in KEYS_A}, val_loss=L0v)]
for e in EPS_:
    for a in ARMS:
        if str(e) in LOG.get(a, {}):
            r = LOG[a][str(e)]
            rows.append(dict(ep=e, arm=a, **{k: r[k] for k in KEYS_A},
                             val_loss=r['diag']['val_loss']))
A = pd.DataFrame(rows)
print(A.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

_e1 = LOG[ARMS[0]].get('1')
if _e1:
    print()
    print('  --- epoch 1 (head FROZEN) da lam gi ---')
    for k in KEYS_A:
        d = _e1[k] - M0[k]
        print(f'    {k:<16} ep0 {M0[k]:.4f} -> ep1 {_e1[k]:.4f}   {d:+.4f}')
    _dm = _e1['macro_map'] - M0['macro_map']
    print()
    if _dm < -0.0015:
        print(f'    => ep1 LAM HAI macro {_dm:+.4f}. Ung ho gia thuyet representation-head mismatch.')
    elif abs(_dm) <= 0.0015:
        print(f'    => ep1 gan nhu KHONG doi gi ({_dm:+.4f}). 1.4h GPU khong mua duoc gi;')
        print(f'       khong the ket luan la "hai", nhung head-freeze cung khong mang lai gi.')
    else:
        print(f'    => ep1 CO ICH ({_dm:+.4f}). Head-freeze khong gay hai dang ke.')
    _e2 = LOG[ARMS[0]].get('2')
    if _e2:
        print(f'    ep1 -> ep2 (head MO): macro {_e2["macro_map"]-_e1["macro_map"]:+.4f}')
        print(f'       Neu ep1 tao mismatch lon, mo head o ep2 phai HOI PHUC manh.')
        print(f'       Luu y: ep2 cung la epoch dau head co the overfit -> hai hieu ung tron nhau.')

# ================= B · optimization trace =================
print()
print('=== B · optimization trace ===')
B = pd.DataFrame([dict(ep=e, arm=a,
                       train_loss=LOG[a][str(e)]['diag']['train_loss'],
                       val_loss=LOG[a][str(e)]['diag']['val_loss'],
                       gap=LOG[a][str(e)]['diag']['train_val_gap'],
                       lr_tail=LOG[a][str(e)]['diag']['lr_tail'],
                       lr_head=LOG[a][str(e)]['diag']['lr_head'],
                       head_unfrozen=LOG[a][str(e)]['diag']['head_unfrozen'],
                       **LOG[a][str(e)]['diag']['drift'],
                       feat_cos=LOG[a][str(e)]['diag'].get('feat_cos_init'),
                       feat_var=LOG[a][str(e)]['diag'].get('feat_var'),
                       rank_pr=LOG[a][str(e)]['diag'].get('feat_rank_pr'))
                  for e in EPS_ for a in ARMS if str(e) in LOG.get(a, {})])
print(B.to_string(index=False, float_format=lambda v: f'{v:.6g}'))

print()
print('  --- chieu LR toan bo 8 epoch (tinh tu cong thuc, khong can du lieu) ---')
_spe = math.ceil(len(IDX['train'])/BATCH/ACCUM)
_TOT = EPOCHS*_spe; _W = max(1, int(WARMUP_FRAC*_TOT))
_cw = lambda s: s/_W if s < _W else 0.5*(1+math.cos(math.pi*(s-_W)/max(_TOT-_W, 1)))
LRP = pd.DataFrame([dict(ep=e, step_end=e*_spe,
                         lr_tail=LR_BLOCKS*_cw(e*_spe), lr_head=LR_HEAD,
                         head_over_tail=(LR_HEAD/max(LR_BLOCKS*_cw(e*_spe), 1e-30)))
                    for e in range(1, EPOCHS+1)])
print(f'  {_spe:,} opt-step/epoch | total {_TOT:,} | warmup {_W:,}')
print(LRP.to_string(index=False, float_format=lambda v: f'{v:.4g}'))
print()
print('  Tail dung warmup+cosine, head dung CONSTANT -> ty so head/tail tang don dieu.')
print('  Cac epoch cuoi tro thanh head-only fine-tuning trong khi representation gan nhu dung.')
print('  Day la lua chon cua toi (thua tu nb12/13), KHONG phai quy dinh cua plan §6 —')
print('  §6 chi khoa lr=1e-5 cho ca hai nhom, khong noi gi ve schedule.')

# ================= C · AP tung nhan + D1/D2 tung nhan =================
print()
print('=== C · AP tung nhan + D1/D2 tung nhan ===')
GCAL = pd.read_csv(f'{SC_DIR}/metrics/gradient_calibration.csv').set_index('label')
for e in EPS_:
    if not all(str(e) in LOG.get(a, {}) for a in ARMS): continue
    ap = {a: np.array(LOG[a][str(e)]['per_ap']) for a in ARMS}
    T = pd.DataFrame(dict(label=LABELS, tier=TIER,
                          **{f'AP_{a}': ap[a] for a in ARMS},
                          D1=ap['R1GM']-ap['R0'], D2=ap['R2']-ap['R1GM'],
                          beta=[LOCK['beta'][j] for j in range(C)],
                          enrich=[GCAL.loc[l, 'enrich'] if l in GCAL.index else np.nan for l in LABELS],
                          rare3=[l in RARE3 for l in LABELS]))
    print()
    print(f'  --- epoch {e} ---')
    print(T[T.rare3].to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
    _nt = T[~T.rare3]
    print(f'    NonTarget-10 (placebo, beta=1 moi arm): |D1| max {_nt.D1.abs().max():.5f} | '
          f'|D2| max {_nt.D2.abs().max():.5f}  '
          f'{"SACH" if max(_nt.D1.abs().max(), _nt.D2.abs().max()) < 0.002 else "! CO DICH CHUYEN"}')

_last = max(e for e in EPS_ if all(str(e) in LOG.get(a, {}) for a in ARMS))
_ap = {a: np.array(LOG[a][str(_last)]['per_ap']) for a in ARMS}
_d2 = (_ap['R2']-_ap['R1GM'])[RARE3_IDX]
_en = np.array([GCAL.loc[LABELS[j], 'enrich'] for j in RARE3_IDX])
print()
print(f'  === gia thuyet: D2 tung nhan di cung chieu voi enrich? (epoch {_last}) ===')
for j, d, en in zip(RARE3_IDX, _d2, _en):
    print(f'    {LABELS[j]:<16} enrich {en:.3f}x   D2 {d:+.5f}')
_ord_en = np.argsort(-_en); _ord_d2 = np.argsort(-_d2)
print(f'    thu tu theo enrich: {[LABELS[RARE3_IDX[i]] for i in _ord_en]}')
print(f'    thu tu theo D2    : {[LABELS[RARE3_IDX[i]] for i in _ord_d2]}')
print(f'    -> {"TRUNG thu tu" if list(_ord_en) == list(_ord_d2) else "KHAC thu tu"}  '
      f'(n=3, day chi la quan sat mo ta, KHONG phai kiem dinh)')

json.dump(dict(seed=SEED, ep0=M0, ep0_val_loss=L0v, epochs=EPS_,
               d2_last_epoch={LABELS[j]: float(d) for j, d in zip(RARE3_IDX, _d2)},
               enrich={LABELS[j]: float(e) for j, e in zip(RARE3_IDX, _en)},
               lr_projection=LRP.to_dict('records'),
               note='read-only diagnostic; khong doi log/lock/state'),
          open(DIAG_JS, 'w'), indent=1)
print()
print(f'-> {DIAG_JS}')
print()
print('=== TIEU CHI (dat TRUOC khi doc so) de goi day la OPTIMIZATION FAILURE ===')
print('  can CA HAI:')
print('    (1) ep1 macro <= ep0 macro - 0.0015   (epoch 1 lam hai, khong phai chi vo ich)')
print('    (2) drift ep1 du lon de goi la doi representation (>= ~0.5% tuong doi)')
print('  Neu (1) sai: head-freeze khong phai thu phai sua.')
print('  Neu (1) dung nhung (2) sai: ep1 hai vi ly do KHAC, khong phai mismatch.')
print()
print('  LUU Y VE TINH LIEM CHINH: doi schedule SAU khi thay ket qua la mot can thiep hau nghiem.')
print('  No cham ca 3 arm nhu nhau nen KHONG doi estimand D2, nhung neu doi roi D2 thanh duong')
print('  thi ta da chon schedule tao ra ket qua duong. Neu doi: ghi la amendment, lock moi,')
print('  va BAO CAO CA HAI run.')

## C15 · Bootstrap `ep0` vs epoch đã train + `D1`/`D2` từng nhãn

**CPU thuần, không GPU.** Chỉ đọc prediction đã lưu (`EP0_s42.npy`, `{arm}_s42_ep{n}.npy`).

Trả lời hai câu:

1. **Ưu thế của `ep0` trên Rare-3 / Tail có thật không**, hay nằm trong nhiễu? `Pleural_Other`
   chỉ có vài trăm positive ở val nên `−0.0027` rất có thể là nhiễu.
2. **Mẫu `D2` từng nhãn** (`PO` dương, `LL` âm, `F` dương nhẹ — ổn định 3/3 epoch) có CI loại
   được 0 không?

Cùng cluster và cùng `N_BOOT` với `C13` (`subject_id`, 2000) để hai bảng đọc chung được.

### Vì sao dùng AP tự cài thay vì `sklearn`

`2000 resample × 10 tập prediction × 13 nhãn = 260,000` lần tính AP. Gọi
`average_precision_score` thì mỗi lần phải `argsort` lại 31k phần tử ⇒ **~40 phút**.

Thay bằng: `argsort` **một lần** cho mỗi (tập, nhãn), rồi mỗi resample chỉ là một vector count
trên thứ tự đã sắp — `bincount` + `cumsum`, không sắp lại. Nhanh hơn **11×**, còn ~4 phút.

⚠ **Phải gộp nhóm điểm bằng nhau.** `sklearn` tính AP theo **nhóm điểm phân biệt**, lấy
precision ở **cuối** mỗi nhóm:

```
AP = sum_g  (a_g / n_pos) * (TP_den_g / N_den_g)
```

Bản đầu tôi tính precision tại **từng bản sao** — sai, vì `k` bản sao của cùng một study
**luôn** tie với nhau. Đo được: **183/300** ca lệch, sai số tới `2.2e-2` trên AP, tức lớn hơn
`D2` đang xét cả một bậc.

Và cái assert đầu tiên tôi viết (so với `sklearn` trên **mẫu đầy đủ**) **không bắt được nó**:
với `c = 1` và điểm liên tục thì không có tie nào, nên bản sai vẫn pass. Gate giờ kiểm với
**count bootstrap thật**, tức đúng chế độ sẽ chạy.

### Đọc kết quả

`ep0` là mốc **chưa train**. `(arm, ep) − ep0` là *"train mua được gì"*.

- CI **chứa 0** ⇒ train không đổi gì đo được trên chỉ số đó
- CI hoàn toàn **âm** ⇒ train **làm hại** — đây là điều đáng kể nhất nếu xảy ra
- CI hoàn toàn **dương** ⇒ train có ích

In [ ]:
BOOT_JS = f'{SC_DIR}/metrics/bootstrap_ep0_seed{SEED}.json'
LOG = json.load(open(LOG_PATH))
EPS_ = sorted({int(e) for e in LOG.get(ARMS[0], {})})
EPS_ = [e for e in EPS_ if all(os.path.exists(pred_p(a, e)) for a in ARMS)]
PS0 = np.load(EP0_NPY)
PRED = {'ep0|INIT': PS0}
for e in EPS_:
    for a in ARMS: PRED[f'ep{e}|{a}'] = np.load(pred_p(a, e))
KEYS = list(PRED)
n_st = len(SID_VAL)
print(f'{len(KEYS)} tap prediction | {n_st:,} study | {len(np.unique(SUBJ_VAL)):,} subject '
      f'| N_BOOT {N_BOOT}')

# ---------------- AP nhanh, XU LY TIE y het sklearn ----------------
# sklearn: AP = sum tren cac NHOM DIEM PHAN BIET cua (dRecall * Precision-cuoi-nhom).
# k ban sao cua cung mot study LUON tie voi nhau -> bat buoc gom nhom, khong duoc
# tinh precision tai tung ban sao.
def _prep(p):
    o = np.argsort(-p, kind='stable'); ps = p[o]
    gid = np.r_[0, np.cumsum(ps[1:] != ps[:-1])].astype('int64')
    return o, gid, int(gid[-1])+1

def ap_grouped(yo, co, gid, ng):
    a = np.bincount(gid, weights=yo*co, minlength=ng)
    n = np.bincount(gid, weights=co,    minlength=ng)
    npos = a.sum()
    if npos <= 0: return np.nan
    TP = np.cumsum(a); N = np.cumsum(n)
    m = a > 0
    return float((a[m]/npos*(TP[m]/N[m])).sum())

PREP = {}; YO = {}
for kk, P in PRED.items():
    pr = [_prep(P[:, j]) for j in range(C)]
    PREP[kk] = pr
    YO[kk] = [Y_VAL[pr[j][0], j].astype('float64') for j in range(C)]

def ap_all(kk, c):
    out = np.full(C, np.nan)
    for j in range(C):
        o, gid, ng = PREP[kk][j]
        out[j] = ap_grouped(YO[kk][j], c[o].astype('float64'), gid, ng)
    return out

# ---------------- GATE ----------------
# CHU Y: kiem voi c = toan 1 la KHONG DU. Khi c toan 1 va diem lien tuc thi khong co
# tie nao, nen mot ban cai SAI ve tie van pass. Phai kiem voi count THAT tu bootstrap.
_ones = np.ones(n_st, 'int64')
_rg = np.random.default_rng(0)
_SU0 = np.unique(SUBJ_VAL); _six = np.searchsorted(_SU0, SUBJ_VAL)
_bad = 0.0
for _t in range(3):
    _c = _ones if _t == 0 else np.bincount(_rg.integers(0, len(_SU0), len(_SU0)),
                                           minlength=len(_SU0))[_six]
    _ix = np.repeat(np.arange(n_st), _c)
    for kk in KEYS:
        _f = ap_all(kk, _c)
        _r = np.array([average_precision_score(Y_VAL[_ix, j], PRED[kk][_ix, j])
                       if Y_VAL[_ix, j].sum() else np.nan for j in range(C)])
        _bad = max(_bad, float(np.nanmax(np.abs(_f-_r))))
    print(f'  gate {_t+1}/3 [{"c=1, KHONG co tie" if _t == 0 else "count bootstrap THAT, CO tie"}]'
          f'  max|diff| {_bad:.2e}')
assert _bad < 1e-9, f'AP tu cai lech sklearn {_bad:.3e} -> DUNG, khong dung ban nhanh'
print(f'AP tu cai == sklearn tren {len(KEYS)} tap x 13 nhan x 3 kieu count  OK')
OBS = {kk: ap_all(kk, _ones) for kk in KEYS}

# ---------------- resample cluster = subject ----------------
SU = np.unique(SUBJ_VAL); n_su = len(SU)
sub_ix = np.searchsorted(SU, SUBJ_VAL)
GRP = dict(macro=np.arange(C), rare3=RARE3_IDX, nontgt=NONTGT_IDX,
           tail=TAIL_IDX, worst3=W3_IDX)
rng = np.random.default_rng(LOCK['seed'])
BOOT = {kk: {g: [] for g in GRP} for kk in KEYS}
BOOT_LAB = {kk: [] for kk in KEYS}
VALID = np.zeros(C); TOT = 0
t0 = time.time()
for b in tqdm(range(N_BOOT), desc='bootstrap (cluster=subject)'):
    c = np.bincount(rng.integers(0, n_su, n_su), minlength=n_su)[sub_ix]
    for kk in KEYS:
        ap = ap_all(kk, c)
        for g, jj in GRP.items(): BOOT[kk][g].append(np.nanmean(ap[jj]))
        BOOT_LAB[kk].append(ap)
    VALID += ~np.isnan(BOOT_LAB[KEYS[0]][-1]); TOT += 1
print(f'xong sau {time.time()-t0:.0f}s')
for kk in KEYS:
    for g in GRP: BOOT[kk][g] = np.asarray(BOOT[kk][g])
    BOOT_LAB[kk] = np.stack(BOOT_LAB[kk])

def ci(v):
    lo, hi = np.percentile(v, [2.5, 97.5]); return float(lo), float(hi)

# ================= 1 · (arm, ep) - ep0 =================
print()
print('=== 1 · TRAIN MUA DUOC GI:  (arm, ep) - ep0   [paired, cluster=subject] ===')
rows = []
for kk in KEYS[1:]:
    e, a = kk.split('|')
    for g, jj in GRP.items():
        d = BOOT[kk][g] - BOOT['ep0|INIT'][g]
        obs = float(np.nanmean(OBS[kk][jj]) - np.nanmean(OBS['ep0|INIT'][jj]))
        lo, hi = ci(d)
        rows.append(dict(ep=e, arm=a, metric=g, observed=obs, ci_lo=lo, ci_hi=hi,
                         excl0=bool(lo > 0 or hi < 0), sign=('+' if lo > 0 else ('-' if hi < 0 else '0'))))
T1 = pd.DataFrame(rows)
for g in GRP:
    print()
    print(f'  --- {g} ---')
    print(T1[T1.metric == g].drop(columns='metric').to_string(index=False,
          float_format=lambda v: f'{v:+.5f}'))
T1.to_csv(f'{SC_DIR}/metrics/bootstrap_ep0_seed{SEED}.csv', index=False)

print()
print('  === PHAN QUYET: train co lam hai Rare-3 / Tail khong ===')
for g in ('rare3', 'tail', 'worst3', 'macro', 'nontgt'):
    sub = T1[T1.metric == g]
    neg = sub[sub.sign == '-']; pos = sub[sub.sign == '+']
    print(f'    {g:<8} CI loai 0 ve phia AM: {len(neg)}/{len(sub)} | '
          f've phia DUONG: {len(pos)}/{len(sub)} | chua 0: {len(sub)-len(neg)-len(pos)}/{len(sub)}')
_r3 = T1[(T1.metric == 'rare3')]
if (_r3.sign == '-').all():
    print('    -> MOI (arm, ep) deu LAM HAI Rare-3 voi CI loai 0. Phat hien dang ke.')
elif (_r3.sign == '0').all():
    print('    -> KHONG (arm, ep) nao khac ep0 tren Rare-3 mot cach do duoc.')
    print('       Uu the cua ep0 nam TRONG nhieu -> KHONG duoc noi "train lam hai Rare-3".')
    print('       Ket luan dung: 8 epoch partial FT KHONG cai thien Rare-3.')
else:
    print('    -> hon hop; doc bang tren, khong tom tat mot cau.')

# ================= 2 · D1 / D2 tung nhan =================
print()
print('=== 2 · D1 / D2 tung nhan Rare-3  [paired, cluster=subject] ===')
rows = []
for e in EPS_:
    for d, hi_, lo_ in (('D1', f'ep{e}|R1GM', f'ep{e}|R0'), ('D2', f'ep{e}|R2', f'ep{e}|R1GM')):
        for j in RARE3_IDX:
            v = BOOT_LAB[hi_][:, j] - BOOT_LAB[lo_][:, j]
            obs = float(OBS[hi_][j] - OBS[lo_][j]); l, h = ci(v)
            rows.append(dict(ep=e, delta=d, label=LABELS[j], observed=obs, ci_lo=l, ci_hi=h,
                             excl0=bool(l > 0 or h < 0), p_gt0=float((v > 0).mean())))
    for d, hi_, lo_ in (('D1', f'ep{e}|R1GM', f'ep{e}|R0'), ('D2', f'ep{e}|R2', f'ep{e}|R1GM')):
        v = BOOT[hi_]['rare3'] - BOOT[lo_]['rare3']
        obs = float(np.nanmean(OBS[hi_][RARE3_IDX]) - np.nanmean(OBS[lo_][RARE3_IDX]))
        l, h = ci(v)
        rows.append(dict(ep=e, delta=d, label='<Rare-3 mAP>', observed=obs, ci_lo=l, ci_hi=h,
                         excl0=bool(l > 0 or h < 0), p_gt0=float((v > 0).mean())))
T2 = pd.DataFrame(rows)
print(T2.to_string(index=False, float_format=lambda v: f'{v:+.5f}'))
T2.to_csv(f'{SC_DIR}/metrics/bootstrap_d1d2_perlabel_seed{SEED}.csv', index=False)

print()
print('  === mau D2 co on dinh va co loai duoc 0 khong ===')
for l in RARE3:
    s = T2[(T2.delta == 'D2') & (T2.label == l)]
    sg = set(np.sign(s.observed.round(6)))
    print(f'    {l:<16} dau {"ON DINH" if len(sg) == 1 else "DOI DAU"} qua {len(s)} epoch | '
          f'CI loai 0: {int(s.excl0.sum())}/{len(s)} epoch | '
          f'observed {[f"{v:+.4f}" for v in s.observed]}')
_agg = T2[(T2.delta == 'D2') & (T2.label == '<Rare-3 mAP>')]
print(f'    <Rare-3 mAP>     CI loai 0: {int(_agg.excl0.sum())}/{len(_agg)} epoch  '
      f'(cong §10 doi >= +0.002; quan sat {[f"{v:+.4f}" for v in _agg.observed]})')

print()
print('=== 3 · valid fraction (§9) ===')
VF = pd.Series(VALID/max(TOT, 1), index=LABELS)
_low = VF[VF < 0.999]
print(f'  nhan co resample thieu positive: {dict(_low.round(4)) if len(_low) else "khong co"}')

json.dump(dict(seed=SEED, n_boot=N_BOOT, cluster='subject_id', n_subject=int(n_su),
               ep0=OBS['ep0|INIT'].tolist(), keys=KEYS,
               vs_ep0=T1.to_dict('records'), d1d2_per_label=T2.to_dict('records'),
               valid_fraction={l: float(v) for l, v in VF.items()},
               note='CPU only; doc prediction da luu; khong doi log/lock/state'),
          open(BOOT_JS, 'w'), indent=1)
print()
print(f'-> {BOOT_JS}')
print()
print('  LUU Y: day la CI cua bien thien LAY MAU tren VAL, KHONG phai bien thien qua SEED.')
print('  §10 van doi cung chieu >= 2/3 seed — CI hep o mot seed khong thay the dieu do.')